# Wheat-rust segmentation on the public NWRD release: one model, one scale

Trains the model named in `MODEL` (next cell) at the scale named in `PRESET` under the protocol shared by every model of the study: CANet-B4 (ours), the NWRD paper's UNet architecture and DeepLabV3+ R50. Three seeds; the checkpoint and the threshold are chosen on the 8 validation photographs; the 10 test photographs are scored once, every pixel at native resolution. The three notebooks (`canet_b4`, `unet`, `deeplabv3plus_r50`) are identical except for `MODEL`, and `scripts/run_on_kaggle.py --preset 128px` sets `PRESET` on the copy it pushes.

**The data.** The public NWRD release (Anwar and Shafait, Zenodo doi:10.5281/zenodo.21672928, CC BY 4.0), attached as the Kaggle dataset `bcap52/nwrd-wheat`. Before anything else, each of its 200 files is checked against its MD5 in `data/manifest.json`, taken from the Zenodo archive, and the run stops on any difference. Windows are cut on a stride fitted to each photograph's size, without padding: 512px windows at native resolution, or 128px windows of the photograph halved with Lanczos (the paper's scale). The training windows go to local disk and are cut again, identically, in every session.

**Kaggle setup**
1. *Add Input* → dataset **`bcap52/nwrd-wheat`**.
2. *Settings* → Accelerator **GPU T4** (one GPU is used), **Internet on** (pip and ImageNet weights).
3. *Save Version* → **Save & Run All (Commit)**. A session runs for up to 12 h.

**If a session stops at its time budget** (the log ends in `INCOMPLETE`), continue it with `python scripts/run_on_kaggle.py --model <model> [--preset 128px] --resume --watch` from the repository, or add this notebook's latest version as an input and commit again. Repeat until the log has no `INCOMPLETE`.

Estimated T4 time for three seeds of 30 epochs: at 512px CANet-B4 3.0 h, UNet 6.2 h, DeepLabV3+ R50 2.0 h; at 128px 2.6, 1.5 and 1.0 h. Each fits in one session.

In [ ]:
import glob, json, os, shutil, subprocess, sys
!pip install -q segmentation-models-pytorch==0.5.0 efficientnet_pytorch==0.7.1 "albumentations>=2.0,<3"
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'enable a GPU accelerator'
os.makedirs('code/nwrd_public', exist_ok=True)
os.makedirs('code/data', exist_ok=True)

In [ ]:
# ── Run settings ────────────────────────────────────────────────────────────────
MODEL = 'deeplabv3plus_b4'   # canet_b4 (ours), unet or deeplabv3plus_r50
PRESET = '512px'   # 512px (native resolution) or 128px (the paper's scale): nwrd_public/config.py
NUM_WORKERS = 4
# Kaggle kills a session at 12 h. Lower this when the weekly GPU quota has less than ~12 h left,
# so the session stops cleanly (and can be resumed) instead of being killed.
# scripts/run_on_kaggle.py --budget-hours sets it on the pushed copy.
KAGGLE_TIME_BUDGET_HOURS = 11.0
# True = this session must continue earlier runs; stop at once if none are attached rather than
# silently retraining everything. scripts/run_on_kaggle.py --resume sets it on the pushed copy.
EXPECT_RESTORE = False

In [ ]:
# Runs go to /kaggle/working/runs, which Kaggle keeps with each committed version.
OUT = '/kaggle/working/runs' if os.path.isdir('/kaggle/working') else os.path.abspath('runs')
os.makedirs(OUT, exist_ok=True)
print(MODEL, PRESET, '| runs ->', OUT, '| time budget', KAGGLE_TIME_BUDGET_HOURS, 'h')

### Pipeline source (generated by `scripts/build_notebooks.py`; edit the repository)

In [ ]:
%%writefile code/data/manifest.json
{
 "files": {
  "test/images/105.jpg": {
   "bytes": 6305393,
   "md5": "990be974d7f9c37f66df2e28dda3959d"
  },
  "test/images/123.jpg": {
   "bytes": 4965109,
   "md5": "d12b781908d98e3dd6f4aaad981c31c4"
  },
  "test/images/25.jpg": {
   "bytes": 6951453,
   "md5": "7340a19abc84fc93a7f95342ead1acff"
  },
  "test/images/29.jpg": {
   "bytes": 7540972,
   "md5": "8bb2248337224796681107afa3190ee5"
  },
  "test/images/62.jpg": {
   "bytes": 6709481,
   "md5": "c9cb6646fd30fa47a871f3eeae0cd827"
  },
  "test/images/63.jpg": {
   "bytes": 7456761,
   "md5": "6d0e0a8d7d625b1b9433fc6a5babc577"
  },
  "test/images/67.jpg": {
   "bytes": 2855227,
   "md5": "0859ae2fd5ce390cafd765b472bcf5c8"
  },
  "test/images/77.jpg": {
   "bytes": 5607492,
   "md5": "383c89e68e3d968cde95d6e7af3f9f2d"
  },
  "test/images/80.jpg": {
   "bytes": 4702887,
   "md5": "4d02db184934acc317be1c72f3daabbb"
  },
  "test/images/89.jpg": {
   "bytes": 6259674,
   "md5": "201a884a11638d71bfd6d62b3d6dc386"
  },
  "test/masks/105.png": {
   "bytes": 31451,
   "md5": "2130a274ef1c20d80203f1284d17631c"
  },
  "test/masks/123.png": {
   "bytes": 35764,
   "md5": "e8e5d0eff55e8afbe772f7b2fe410812"
  },
  "test/masks/25.png": {
   "bytes": 37330,
   "md5": "9721055c349f9039a71e48d60c6085f9"
  },
  "test/masks/29.png": {
   "bytes": 39133,
   "md5": "2bf1b420d7491ce45cf364ecfaf958b4"
  },
  "test/masks/62.png": {
   "bytes": 82201,
   "md5": "b4ab5bddf0ba74e04ba504fe027b8065"
  },
  "test/masks/63.png": {
   "bytes": 78220,
   "md5": "4c94b13d99123aafc668b2bd97517496"
  },
  "test/masks/67.png": {
   "bytes": 42593,
   "md5": "a472e3559092ab6bca06d17a248b849c"
  },
  "test/masks/77.png": {
   "bytes": 25252,
   "md5": "8f85dcb3d40d057428b0f203192321fb"
  },
  "test/masks/80.png": {
   "bytes": 27139,
   "md5": "068b38406b1c3dd34dd58d73ba117ac0"
  },
  "test/masks/89.png": {
   "bytes": 35785,
   "md5": "e38171076df6bfe4f1129f0b0aabf86a"
  },
  "train/images/10.jpg": {
   "bytes": 6946041,
   "md5": "b484079929b37e3d956cbbef99fd8319"
  },
  "train/images/100.jpg": {
   "bytes": 2992082,
   "md5": "88fc282068725ac1f16be7100c0526f9"
  },
  "train/images/101.jpg": {
   "bytes": 5978822,
   "md5": "645a643ebe205b9f5c130c5006275b81"
  },
  "train/images/102.jpg": {
   "bytes": 5791698,
   "md5": "ef66e75ac792fc8f7c159acc67220dea"
  },
  "train/images/103.jpg": {
   "bytes": 6178339,
   "md5": "0b1d240fbec97d5b0539a58d26f48447"
  },
  "train/images/104.jpg": {
   "bytes": 7771372,
   "md5": "31d7dd388e858318717dcd07e045b92e"
  },
  "train/images/109.jpg": {
   "bytes": 5942912,
   "md5": "08a19b47343354ea10d42e4fafef815a"
  },
  "train/images/11.jpg": {
   "bytes": 7218585,
   "md5": "33205cb1687d3f16ededbe6f21fb3dac"
  },
  "train/images/110.jpg": {
   "bytes": 4669777,
   "md5": "f9e5c24f97ba08cfcc73dc4717e8e7db"
  },
  "train/images/111.jpg": {
   "bytes": 4780116,
   "md5": "554e26f1d13b2a4e75b2cd651a21ae13"
  },
  "train/images/113.jpg": {
   "bytes": 5564855,
   "md5": "a4891cdca50fc87759eb29fd5a0e9b5e"
  },
  "train/images/114.jpg": {
   "bytes": 4960698,
   "md5": "427e93cec2ead020392e78e0820ef015"
  },
  "train/images/117.jpg": {
   "bytes": 5396788,
   "md5": "2f4b6336ba0e85b2aa425d4bdb5dc1d2"
  },
  "train/images/118.jpg": {
   "bytes": 5285925,
   "md5": "2b9b3e539435a6d4aa8e42ff076d724f"
  },
  "train/images/119.jpg": {
   "bytes": 5290228,
   "md5": "ab26952cd220a90f46fdf696917746d5"
  },
  "train/images/12.jpg": {
   "bytes": 7649224,
   "md5": "338a1a8a59f9ec4ac61c7643d6c98489"
  },
  "train/images/120.jpg": {
   "bytes": 6978177,
   "md5": "1c07e382f3a366516f376bc4c37db2b7"
  },
  "train/images/121.jpg": {
   "bytes": 5754762,
   "md5": "2bc25d42b9fbe7b5dd5cecd60142f579"
  },
  "train/images/122.jpg": {
   "bytes": 6866193,
   "md5": "4f9fda19509cc7b0ea098d57b9b0d344"
  },
  "train/images/124.jpg": {
   "bytes": 5364228,
   "md5": "77be1c9f420b957ed96ce68520412db5"
  },
  "train/images/125.jpg": {
   "bytes": 6293282,
   "md5": "05c36dbe7fc159e3516e3f17da41098e"
  },
  "train/images/128.jpg": {
   "bytes": 5828144,
   "md5": "408069506846964ceb31313ce12b80df"
  },
  "train/images/129.jpg": {
   "bytes": 5854195,
   "md5": "f0eeca018b3d8dff34f8c9ed4c1949aa"
  },
  "train/images/13.jpg": {
   "bytes": 6928129,
   "md5": "3fa0fad827c97f6bcf8077c969aff8e2"
  },
  "train/images/130.jpg": {
   "bytes": 5927118,
   "md5": "dcaa6b6547cca19a7972612e3a9e63cf"
  },
  "train/images/131.jpg": {
   "bytes": 6305409,
   "md5": "ecfcf92002fd624256b47c691acd8656"
  },
  "train/images/132.jpg": {
   "bytes": 4589223,
   "md5": "13a091a9d97d4f3f14bda58ccbb6f983"
  },
  "train/images/133.jpg": {
   "bytes": 4288930,
   "md5": "d5fd8146fef165543d7eaf4a1e750041"
  },
  "train/images/134.jpg": {
   "bytes": 6489508,
   "md5": "40eac5d456b2b5805f13c34a16a0aca4"
  },
  "train/images/135.jpg": {
   "bytes": 6474807,
   "md5": "5fd6260e48de193e48a3f43acc77d4a1"
  },
  "train/images/136.jpg": {
   "bytes": 2901579,
   "md5": "032a76537614557cafe5b70a2c40fc3c"
  },
  "train/images/137.jpg": {
   "bytes": 2992082,
   "md5": "1fae0f492c1d22895abe5a9b9a19ae9c"
  },
  "train/images/138.jpg": {
   "bytes": 2736717,
   "md5": "834535249d380daba885b177705c9dbe"
  },
  "train/images/139.jpg": {
   "bytes": 5146248,
   "md5": "ada4c73ddca0b02372ff9c15f3ace1e4"
  },
  "train/images/14.jpg": {
   "bytes": 6329738,
   "md5": "b252205b9d47d2ce921ea986e2c32f1a"
  },
  "train/images/140.jpg": {
   "bytes": 4785828,
   "md5": "20eff414c9a99a3274efb6b447e52bdb"
  },
  "train/images/141.jpg": {
   "bytes": 6145465,
   "md5": "e713dd9d87f6060ce55c3e848568d462"
  },
  "train/images/142.jpg": {
   "bytes": 5833644,
   "md5": "3239a6aec4aef275a9e6e71e53564bba"
  },
  "train/images/144.jpg": {
   "bytes": 5913765,
   "md5": "a833639cd6b66af71c59008df6b1edfc"
  },
  "train/images/145.jpg": {
   "bytes": 5431465,
   "md5": "c44825346c80826557e672224851e709"
  },
  "train/images/146.jpg": {
   "bytes": 3327563,
   "md5": "eb6553bf8425f415167a1c7d477f822c"
  },
  "train/images/149.jpg": {
   "bytes": 5683193,
   "md5": "5d84fc6e57cd421d15e20f8a6cf8e25f"
  },
  "train/images/15.jpg": {
   "bytes": 7399326,
   "md5": "4a2aa8015aac60357307f093fa585932"
  },
  "train/images/150.jpg": {
   "bytes": 6592135,
   "md5": "5dd2a8afe4e4fad90fc26c8d53f46fa8"
  },
  "train/images/19.jpg": {
   "bytes": 7554806,
   "md5": "f91de2bf41d1beb61ebcb67c3e5d53a0"
  },
  "train/images/2.jpg": {
   "bytes": 6615038,
   "md5": "b2b90885599aeeff3511b703133e38b3"
  },
  "train/images/21.jpg": {
   "bytes": 6431028,
   "md5": "cce60324eac2af6e0a5cfd01e4cfe448"
  },
  "train/images/22.jpg": {
   "bytes": 6423913,
   "md5": "ae79f90c2576b81e070bb12ea58c27c5"
  },
  "train/images/24.jpg": {
   "bytes": 7181269,
   "md5": "44c51e813d5c4fe13bae0e85a0d1c3c5"
  },
  "train/images/26.jpg": {
   "bytes": 7599003,
   "md5": "e4db2b26adae1007a726b498b22ceeb0"
  },
  "train/images/27.jpg": {
   "bytes": 7264172,
   "md5": "be3254fac1cb1d209b15a6456e6ab7d6"
  },
  "train/images/28.jpg": {
   "bytes": 7274747,
   "md5": "f3d8462d5b41866db4a189cba3cc1240"
  },
  "train/images/3.jpg": {
   "bytes": 7702174,
   "md5": "30fd9afe7d135d461ba397e8c57d6f3e"
  },
  "train/images/30.jpg": {
   "bytes": 6200337,
   "md5": "95d368d70f994cc25f8bdb704606365b"
  },
  "train/images/31.jpg": {
   "bytes": 6919495,
   "md5": "95beb690f5fee1bbb3d1ead7703734a5"
  },
  "train/images/32.jpg": {
   "bytes": 7399326,
   "md5": "4a2aa8015aac60357307f093fa585932"
  },
  "train/images/33.jpg": {
   "bytes": 7706034,
   "md5": "2bdf56aeea8028b0babb1b769e097725"
  },
  "train/images/5.jpg": {
   "bytes": 6731411,
   "md5": "f8d1541e7af618f43bb52a7771a3a4b9"
  },
  "train/images/57.jpg": {
   "bytes": 6733781,
   "md5": "34d92dfbbf795c6af3297bafdcd3150d"
  },
  "train/images/58.jpg": {
   "bytes": 6877585,
   "md5": "e47f00addfa82eaf09017916899c871a"
  },
  "train/images/60.jpg": {
   "bytes": 6322926,
   "md5": "a3c3f16f4ad180252bd912c00802be9c"
  },
  "train/images/61.jpg": {
   "bytes": 6898132,
   "md5": "1aba7899494d24ee6c572e3cbc540e53"
  },
  "train/images/64.jpg": {
   "bytes": 7403875,
   "md5": "28a1a059b79be0d59a11a403ecbc015f"
  },
  "train/images/69.jpg": {
   "bytes": 6145465,
   "md5": "7a6018e94bee3b0db0d57453648b87ed"
  },
  "train/images/7.jpg": {
   "bytes": 6104866,
   "md5": "0f0f2d47a5003d9b99ceb352642cc46c"
  },
  "train/images/71.jpg": {
   "bytes": 5902392,
   "md5": "ddaaea88cc853dbc73dd94933b2775f9"
  },
  "train/images/72.jpg": {
   "bytes": 6355832,
   "md5": "0d29f4f33d77ddadf81742c6d449f0fd"
  },
  "train/images/73.jpg": {
   "bytes": 7063729,
   "md5": "3ba7fdf5d6cbcf3681c607bb268b062d"
  },
  "train/images/74.jpg": {
   "bytes": 6948707,
   "md5": "3011812b6dfe87e3b77b6b703976fbed"
  },
  "train/images/75.jpg": {
   "bytes": 5799662,
   "md5": "040c124eac8921d06d87ee9ff4f5b201"
  },
  "train/images/76.jpg": {
   "bytes": 5582334,
   "md5": "b7296383c8f413b59d5fa51babef958a"
  },
  "train/images/78.jpg": {
   "bytes": 6694605,
   "md5": "da45a38f2947e3064351b7cf9b7db48f"
  },
  "train/images/79.jpg": {
   "bytes": 4702887,
   "md5": "64ee8f93a32bca3502d38a9d192e5056"
  },
  "train/images/81.jpg": {
   "bytes": 6694605,
   "md5": "1d7c69f11880a18240b61de7886fbb7d"
  },
  "train/images/83.jpg": {
   "bytes": 5828144,
   "md5": "211eebcfa6e37dd59823c71a0b502918"
  },
  "train/images/84.jpg": {
   "bytes": 7568909,
   "md5": "75c171cf8a4c82d2a10d53b704abd5bd"
  },
  "train/images/85.jpg": {
   "bytes": 7352778,
   "md5": "13d2601f136d0d3ee83cfaf23b328bbc"
  },
  "train/images/86.jpg": {
   "bytes": 6259674,
   "md5": "8e797b5759da1aedc83fca61deb45d0c"
  },
  "train/images/87.jpg": {
   "bytes": 6492966,
   "md5": "6f2898500fdd44bef331ed3db4c356f7"
  },
  "train/images/88.jpg": {
   "bytes": 6550103,
   "md5": "624c4cba3b95698f42a61a15632976db"
  },
  "train/images/9.jpg": {
   "bytes": 7586002,
   "md5": "4c81e3ea62216395ad3490091ab0de77"
  },
  "train/images/90.jpg": {
   "bytes": 5087275,
   "md5": "bbdf38f9e3a245e6ad6673f727dbca86"
  },
  "train/images/91.jpg": {
   "bytes": 7446319,
   "md5": "80acb4b26b2b9f7b5a34689264097235"
  },
  "train/images/92.jpg": {
   "bytes": 5262558,
   "md5": "92f9f7fd089bfef8524ad07a209233c4"
  },
  "train/images/93.jpg": {
   "bytes": 5205929,
   "md5": "bed5b29803ec82c0cd21b099bc17c5f8"
  },
  "train/images/94.jpg": {
   "bytes": 3990348,
   "md5": "2da743a4e0297f90b35e7b076958a077"
  },
  "train/images/95.jpg": {
   "bytes": 5577158,
   "md5": "58b560f4c1d20e700d16b2974c7d34c3"
  },
  "train/images/96.jpg": {
   "bytes": 6492966,
   "md5": "8aae53b7abe7206c0e4fa9d1d38a2f49"
  },
  "train/images/98.jpg": {
   "bytes": 7446319,
   "md5": "a5136eb7e830b36ca6e59ec8292bac73"
  },
  "train/images/99.jpg": {
   "bytes": 6822549,
   "md5": "02ab750de2ff3ad76f6b7eaf5fd5d3b5"
  },
  "train/masks/10.png": {
   "bytes": 40087,
   "md5": "06ded56619dae9c03c6c9b8583635929"
  },
  "train/masks/100.png": {
   "bytes": 25710,
   "md5": "40f33bde427fcfc11fd53a1c3d3eac98"
  },
  "train/masks/101.png": {
   "bytes": 27223,
   "md5": "1adcf8543db73c08621e9e3f2ec73300"
  },
  "train/masks/102.png": {
   "bytes": 23848,
   "md5": "6d08e63480aba6f90e214be343c88626"
  },
  "train/masks/103.png": {
   "bytes": 21574,
   "md5": "fdaef50372105f68d1962c8368c61954"
  },
  "train/masks/104.png": {
   "bytes": 30066,
   "md5": "d3c9de5c3e760e315ed904d8c802ef29"
  },
  "train/masks/109.png": {
   "bytes": 46879,
   "md5": "527d4d5081dc606cb8b67cc53f971095"
  },
  "train/masks/11.png": {
   "bytes": 33229,
   "md5": "f4a6d685f76e79878811121906342b51"
  },
  "train/masks/110.png": {
   "bytes": 30770,
   "md5": "859e5ec44d4a42ea71d047f6b666cf15"
  },
  "train/masks/111.png": {
   "bytes": 36281,
   "md5": "f3acae94811923538f2750a080d6ff00"
  },
  "train/masks/113.png": {
   "bytes": 29636,
   "md5": "5999b205cab4e085499da400d74a0a86"
  },
  "train/masks/114.png": {
   "bytes": 25078,
   "md5": "791f9ea5bf85ad02f5f3ec12778fa823"
  },
  "train/masks/117.png": {
   "bytes": 49446,
   "md5": "410621ca60e689eaa912fb9aa9e2e0bb"
  },
  "train/masks/118.png": {
   "bytes": 28266,
   "md5": "0fc3494836c30be5379b250033f15b34"
  },
  "train/masks/119.png": {
   "bytes": 30707,
   "md5": "1f02faf027024e6bdf7063ec8aaa04a2"
  },
  "train/masks/12.png": {
   "bytes": 31388,
   "md5": "87140cf68fb3e311ff09281e34b5170e"
  },
  "train/masks/120.png": {
   "bytes": 30789,
   "md5": "c3b34bc96b910f3ae0dd08e7f0635579"
  },
  "train/masks/121.png": {
   "bytes": 33246,
   "md5": "79020a8872071e5018491ff634c228d9"
  },
  "train/masks/122.png": {
   "bytes": 28734,
   "md5": "c6dd1a0ff09666983e470f66a0ab4ac4"
  },
  "train/masks/124.png": {
   "bytes": 37261,
   "md5": "70cf24658df3e6ab2ca55729d8bd200e"
  },
  "train/masks/125.png": {
   "bytes": 34455,
   "md5": "20cda3c4e2ab3608d4674eccccdb948d"
  },
  "train/masks/128.png": {
   "bytes": 22241,
   "md5": "589c40a79a4b49d72eebd3c837dbb8a7"
  },
  "train/masks/129.png": {
   "bytes": 29718,
   "md5": "e9182bc9db8fa8e5842a3fa18ebb6211"
  },
  "train/masks/13.png": {
   "bytes": 33263,
   "md5": "5ca0ffe5fed40d72d0236cdcc82c6a37"
  },
  "train/masks/130.png": {
   "bytes": 29589,
   "md5": "2cb2cd5aea32a7ad47c16e34ea96a771"
  },
  "train/masks/131.png": {
   "bytes": 28868,
   "md5": "5d63931367e409c8703fd1045498f8eb"
  },
  "train/masks/132.png": {
   "bytes": 30140,
   "md5": "06083454df2dfd7a5d1cd9840bdb69c3"
  },
  "train/masks/133.png": {
   "bytes": 26848,
   "md5": "c2a835c3115be178cf857670dfea6a09"
  },
  "train/masks/134.png": {
   "bytes": 27136,
   "md5": "4842c6b56473a8626f718f1104c1a630"
  },
  "train/masks/135.png": {
   "bytes": 26192,
   "md5": "5c53dcf3661b534c57f46a2bbfbb9902"
  },
  "train/masks/136.png": {
   "bytes": 25262,
   "md5": "44167f19d08acd495f92c6faff35dc68"
  },
  "train/masks/137.png": {
   "bytes": 27818,
   "md5": "4db4e9e6caa0fc3ab104221e2c35df8b"
  },
  "train/masks/138.png": {
   "bytes": 40175,
   "md5": "ba3599ddb1fdaed736746a87bea54d77"
  },
  "train/masks/139.png": {
   "bytes": 38290,
   "md5": "716da55b17898ce2e4cb22020204a300"
  },
  "train/masks/14.png": {
   "bytes": 34886,
   "md5": "2cf3d49bd927357d0803cf373635118c"
  },
  "train/masks/140.png": {
   "bytes": 29339,
   "md5": "936639af646e4c659370cee3cf5e0f52"
  },
  "train/masks/141.png": {
   "bytes": 29404,
   "md5": "a1940dfc597abf035c2d29fc294d1d12"
  },
  "train/masks/142.png": {
   "bytes": 29364,
   "md5": "b6957d61fcfdbfc536b99ed7a8a51094"
  },
  "train/masks/144.png": {
   "bytes": 25782,
   "md5": "ce12ad3dc93383347f3d9e5bfab6ffb6"
  },
  "train/masks/145.png": {
   "bytes": 32276,
   "md5": "e7159f5fcb0afb24c55f1e691fc3528d"
  },
  "train/masks/146.png": {
   "bytes": 36557,
   "md5": "77f0886def2c2917a491bd246fedc586"
  },
  "train/masks/149.png": {
   "bytes": 22039,
   "md5": "d6528bd4540eb4e8e629c9ff61b30f7a"
  },
  "train/masks/15.png": {
   "bytes": 31854,
   "md5": "4e9c0f4f47bcf3ca4f0e7452e14d98c1"
  },
  "train/masks/150.png": {
   "bytes": 32091,
   "md5": "3b4119c7002c00f2ecdfc3ab358e0129"
  },
  "train/masks/19.png": {
   "bytes": 115690,
   "md5": "6c0e65ca7acfcef2e03fba97fbbc0562"
  },
  "train/masks/2.png": {
   "bytes": 32661,
   "md5": "8a2ec64773bdb4db581330b17a5ddc56"
  },
  "train/masks/21.png": {
   "bytes": 39651,
   "md5": "433d34d2a7213c30c626ab92884bae78"
  },
  "train/masks/22.png": {
   "bytes": 32219,
   "md5": "c27872d6210557bfc396626ab571de87"
  },
  "train/masks/24.png": {
   "bytes": 109902,
   "md5": "f511da5877ae7f24cc0eaf03dd4e07f7"
  },
  "train/masks/26.png": {
   "bytes": 35137,
   "md5": "ff2b0e6de257a115aacdc7775323b469"
  },
  "train/masks/27.png": {
   "bytes": 42885,
   "md5": "b1f9dbbb50cef227680a054a692306d6"
  },
  "train/masks/28.png": {
   "bytes": 47331,
   "md5": "eb2bc30e7c840e6eab0b8ccbd029420a"
  },
  "train/masks/3.png": {
   "bytes": 35790,
   "md5": "90fba275fb1b43be2234c103d5f2c8e2"
  },
  "train/masks/30.png": {
   "bytes": 36540,
   "md5": "c2e2a16cab52f1770c2a5c6dda2ec4a8"
  },
  "train/masks/31.png": {
   "bytes": 34248,
   "md5": "655bb2ca6295f7c8fe5fe688cefe7884"
  },
  "train/masks/32.png": {
   "bytes": 33032,
   "md5": "560199775d8a1799268336476bd0b816"
  },
  "train/masks/33.png": {
   "bytes": 91719,
   "md5": "1ecbc30eff7fcaae4dbfda3899847138"
  },
  "train/masks/5.png": {
   "bytes": 38128,
   "md5": "e489338c706b626874a0a2fd08156481"
  },
  "train/masks/57.png": {
   "bytes": 35893,
   "md5": "0fb1606e39dee50b3ad31c1e7929585e"
  },
  "train/masks/58.png": {
   "bytes": 78463,
   "md5": "75c4158932a225eed334232438bfcb46"
  },
  "train/masks/60.png": {
   "bytes": 47850,
   "md5": "4d09f5ffe505a4910f9f5ff249796fb0"
  },
  "train/masks/61.png": {
   "bytes": 51472,
   "md5": "47cc0657930e869c6ece8da31819f711"
  },
  "train/masks/64.png": {
   "bytes": 72100,
   "md5": "84fedd9cc96531b4b8775e9f82da2ef8"
  },
  "train/masks/69.png": {
   "bytes": 26820,
   "md5": "89b0113bfe6d1631350d83e136a2e0d1"
  },
  "train/masks/7.png": {
   "bytes": 35460,
   "md5": "e99a4fe901302f8dea8233a23208be61"
  },
  "train/masks/71.png": {
   "bytes": 27284,
   "md5": "67e6bbb47c42eb6247134e46cd2c273a"
  },
  "train/masks/72.png": {
   "bytes": 28588,
   "md5": "909fb4d15f6d4a0141ed70ccfe1f4dd4"
  },
  "train/masks/73.png": {
   "bytes": 24954,
   "md5": "f01e09cd72a5fb8ca1fc2d8684c538d1"
  },
  "train/masks/74.png": {
   "bytes": 25460,
   "md5": "5d20efdf281681a2db13b1e15c56b28f"
  },
  "train/masks/75.png": {
   "bytes": 25570,
   "md5": "de54b3945dc179feeb288c3e91dc23e6"
  },
  "train/masks/76.png": {
   "bytes": 28594,
   "md5": "d79ea0a3455b6ca1289524c98b30c0ab"
  },
  "train/masks/78.png": {
   "bytes": 24625,
   "md5": "c7c0e46105a3013ea2f2bbbeb4e07cdf"
  },
  "train/masks/79.png": {
   "bytes": 27047,
   "md5": "71a821d4330bf8d1fe454ba6f23b4e9b"
  },
  "train/masks/81.png": {
   "bytes": 24619,
   "md5": "10a1b1d3cd7f590f89ff5e0b1fdea945"
  },
  "train/masks/83.png": {
   "bytes": 22108,
   "md5": "e341da4196e031fc55520e540f5f8ebb"
  },
  "train/masks/84.png": {
   "bytes": 31063,
   "md5": "3ebe488d85b7f0499f4f1d5cf2e1be64"
  },
  "train/masks/85.png": {
   "bytes": 41168,
   "md5": "11b6f96d6b223fc327b8cdf1d10338c8"
  },
  "train/masks/86.png": {
   "bytes": 36357,
   "md5": "b2652deae3826d1c23f900bdc0c38c68"
  },
  "train/masks/87.png": {
   "bytes": 33432,
   "md5": "6e37ba096e307560c6038f30e9be136d"
  },
  "train/masks/88.png": {
   "bytes": 31927,
   "md5": "1238815a218667344e60a8866a782084"
  },
  "train/masks/9.png": {
   "bytes": 41794,
   "md5": "cc0ae1520983abc4ca61b60f54ef7202"
  },
  "train/masks/90.png": {
   "bytes": 41235,
   "md5": "53ac4fe3fccebb3659ce1bb975d61190"
  },
  "train/masks/91.png": {
   "bytes": 35741,
   "md5": "01447bcb6db0c4fadcd787d7deb35e4d"
  },
  "train/masks/92.png": {
   "bytes": 53008,
   "md5": "6f963300fd78f376fd6c7a5274da9b84"
  },
  "train/masks/93.png": {
   "bytes": 28462,
   "md5": "2651e971c2e154ab5a33575c957fac5f"
  },
  "train/masks/94.png": {
   "bytes": 50722,
   "md5": "5d3a72117d859018ad5d565cff61270f"
  },
  "train/masks/95.png": {
   "bytes": 58641,
   "md5": "a25063d635d80d83c5ce414ad622ce5e"
  },
  "train/masks/96.png": {
   "bytes": 31153,
   "md5": "0a1ecb1477ad069df2b6f88e2a23ae4f"
  },
  "train/masks/98.png": {
   "bytes": 33691,
   "md5": "d2a735af1ef68cb867ff132550fb87c4"
  },
  "train/masks/99.png": {
   "bytes": 25579,
   "md5": "533c86aac7783b7d031b5206fb42983d"
  }
 },
 "source": "NWRD.zip from Zenodo (Anwar and Shafait, doi:10.5281/zenodo.21672928, v1.0, CC BY 4.0), 608,898,312 bytes, MD5 7b9193efebe1ed8c60b10c9f05d11e14"
}


In [ ]:
%%writefile code/nwrd_public/__init__.py
"""CANet-B4 against the NWRD paper's baselines on the public NWRD release."""

__version__ = '1.0.0'


In [ ]:
%%writefile code/nwrd_public/config.py
"""Experiment protocol. Every model is trained and scored with exactly these settings.

Each field is also a command-line flag of `python -m nwrd_public.run` (`data_root` ->
`--data-root`). A preset sets the window size and scale; flags given explicitly override it.
"""

from dataclasses import asdict, dataclass, field

# CANet-B4 is ours; UNet is the NWRD paper's architecture; DeepLabV3+ R50 is the second baseline.
# The decision rule (docs/protocol.md §5) judges CANet-B4 against these two baselines only.
COMPARED_MODELS = ('canet_b4', 'unet', 'deeplabv3plus_r50')
# The exploratory 512px ablation, added 2026-11-15 after the 512px results (docs/protocol.md §9):
# each changes one part of DeepLabV3+ R50 or CANet-B4. models.py also defines NWRD_final's
# `unet_tuned` control (and mentions its config.unet_tuned_lr), which is not run here.
ABLATION_MODELS = ('deeplabv3plus_r50_scratch', 'unet_r50', 'deeplabv3plus_b4')


@dataclass
class Config:
    data_root: str = ''  # holds the release's train/ and test/; '' searches /kaggle/input
    manifest: str = ''  # MD5 manifest of the release; '' is data/manifest.json
    out_dir: str = 'runs'
    model: str = 'canet_b4'
    seeds: list[int] = field(default_factory=lambda: [42, 43, 44])
    data_seed: int = 42  # background subsample; fixed so all runs see the same windows

    # Windows of `window` px are cut from each photograph resized by `scale` (Lanczos), on the
    # fitted grid of nwrd_public/windows.py: no padding, every pixel covered.
    window: int = 512
    scale: float = 1.0
    window_dir: str = ''  # where training windows are cut; default: under the temp directory
    bg_keep_ratio: float = 0.20
    epochs: int = 30
    freeze_encoder_epochs: int = 3
    batch_size: int = 8
    num_workers: int = 2
    lr: float = 5e-5
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    dice_weight: float = 0.7
    bce_weight: float = 0.3
    dropout: float = 0.5  # CANet head dropout; the other models have none
    sched_t0: int = 5
    sched_tmult: int = 2
    sched_eta_min: float = 1e-7
    amp: bool = True
    pretrained: bool = True

    # Threshold chosen on val over this grid, then frozen and applied to test. Rust is sparse in
    # this release, so the BCE rust weight is high and the best threshold may lie above 0.95.
    thr_lo: float = 0.01
    thr_hi: float = 0.99
    thr_step: float = 0.01

    time_budget_hours: float = 0.0  # >0: stop cleanly before a platform session limit
    limit: int = 0  # >0: use only the first N photographs of each split (smoke tests)

    def validate(self) -> None:
        if self.model not in COMPARED_MODELS + ABLATION_MODELS:
            raise ValueError(
                f'model {self.model!r} is not run here; use {COMPARED_MODELS + ABLATION_MODELS}'
            )
        if not 0 < self.scale <= 1:
            raise ValueError(
                f'scale {self.scale} must be in (0, 1]: photographs are never enlarged'
            )
        if self.window <= 0:
            raise ValueError(f'window {self.window} must be positive')

    def to_dict(self) -> dict:
        return asdict(self)


# The two scales. `512px` is native resolution; `128px` is the NWRD paper's scale, 128px windows
# of photographs halved with Lanczos. Everything else is the same, so 512px vs 128px is a
# one-variable comparison.
PRESETS = {
    '512px': {'window': 512, 'scale': 1.0},
    '128px': {'window': 128, 'scale': 0.5},
}


def preset_of(window: int, scale: float) -> str:
    """The preset with this window size and scale."""
    for name, p in PRESETS.items():
        if p['window'] == window and p['scale'] == scale:
            return name
    raise ValueError(f'no preset has window {window} and scale {scale}')


In [ ]:
%%writefile code/nwrd_public/dataset.py
"""The public NWRD release: finding it, checking it against its manifest, the split, and reading
photographs, masks and cut windows.

The release (Anwar and Shafait, Zenodo doi:10.5281/zenodo.21672928, CC BY 4.0) holds 100
photographs of 10 sizes with pixel masks: <root>/{train,test}/{images,masks}/<id>.{jpg,png}.
docs/dataset.md describes it, and docs/protocol.md §2 explains the split.
"""

import glob
import hashlib
import json
import os
from dataclasses import dataclass

import cv2
import numpy as np
import torch

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet statistics
ZENODO_URL = 'https://zenodo.org/api/records/21672928/files/NWRD.zip/content'
ZENODO_MD5 = '7b9193efebe1ed8c60b10c9f05d11e14'
ZENODO_BYTES = 608_898_312

# The authors' validation photographs: their loader shuffles the sorted training file names with
# NumPy seed 0 and keeps the first 10% (paper_validation_ids). Their test set is the test/ folder.
PAPER_VAL_IDS = ('2', '30', '72', '87', '101', '118', '136', '149', '150')
# Photo 101's mask is 19 px narrower and 21 px shorter than the photograph, and lines up with it
# only shifted 8 px left and 1 px up, partly past its edge (docs/dataset.md §3): it is in no split.
EXCLUDED_MISALIGNED = ('101',)
VAL_IDS = tuple(i for i in PAPER_VAL_IDS if i not in EXCLUDED_MISALIGNED)
# The release repeats ten scenes: nine pairs of files decode to identical pixels, and 149 is 99
# cropped and saved again; each copy has its own mask. These four training photographs are twins
# of validation or test photographs (value), so no model trains on them.
EXCLUDED_TWINS = {'79': '80', '86': '89', '96': '87', '99': '149'}
TWIN_PAIRS = (
    ('15', '32'),
    ('83', '128'),
    ('100', '137'),
    ('69', '141'),
    ('78', '81'),
    ('91', '98'),
    ('79', '80'),
    ('86', '89'),
    ('96', '87'),
    ('99', '149'),
)


def default_manifest_path() -> str:
    """data/manifest.json next to the package, in the repository and in a notebook's code/."""
    return os.path.join(
        os.path.dirname(os.path.dirname(os.path.abspath(__file__))), 'data', 'manifest.json'
    )


def load_manifest(path: str | None = None) -> dict:
    """{'source': ..., 'files': {'train/images/10.jpg': {'md5': ..., 'bytes': ...}, ...}}"""
    with open(path or default_manifest_path(), encoding='utf-8') as f:
        return json.load(f)


def md5_of(path: str) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def photo_id(name: str) -> str:
    return os.path.splitext(os.path.basename(name))[0]


def photo_folders(manifest: dict) -> dict[str, str]:
    """Photograph id -> the release folder holding it, 'train' or 'test'."""
    return {photo_id(n): n.split('/')[0] for n in manifest['files'] if '/images/' in n}


def paper_validation_ids(train_images: list[str]) -> list[str]:
    """The authors' validation draw (`BaseDataLoader._split_sampler`, validation_split 0.1) over
    the training images' paths sorted as text. The legacy global NumPy generator after
    np.random.seed(0) draws exactly what RandomState(0) does."""
    names = sorted(train_images)
    idx = np.arange(len(names))
    np.random.RandomState(0).shuffle(idx)
    return [photo_id(names[i]) for i in idx[: int(len(names) * 0.1)]]


def splits(manifest: dict) -> dict[str, list[str]]:
    """Photograph ids per split, each sorted by number: the authors' validation and test
    photographs, and their training photographs without EXCLUDED_TWINS; photo 101
    (EXCLUDED_MISALIGNED) in none."""
    folders = photo_folders(manifest)
    stray = [i for i in PAPER_VAL_IDS if folders.get(i) != 'train']
    if stray:
        raise ValueError(f'validation photographs {stray} are not in the release train/ folder')
    held_out = set(PAPER_VAL_IDS) | set(EXCLUDED_TWINS) | set(EXCLUDED_MISALIGNED)
    in_train = [i for i, f in folders.items() if f == 'train']
    return {
        'train': sorted((i for i in in_train if i not in held_out), key=int),
        'val': sorted(VAL_IDS, key=int),
        'test': sorted((i for i, f in folders.items() if f == 'test'), key=int),
    }


def find_release(hint: str | None, manifest: dict) -> str:
    """The folder holding the release's train/ and test/: `hint` itself, or the first folder
    below it, then below Kaggle's /kaggle/input, where the manifest's first file sits."""
    probe = sorted(manifest['files'])[0].split('/')
    candidates = [hint] if hint else []
    candidates.append('/kaggle/input')
    for base in candidates:
        if os.path.isfile(os.path.join(base, *probe)):
            return base
        hits = sorted(glob.glob(os.path.join(glob.escape(base), '**', *probe), recursive=True))
        if hits:
            root = hits[0]
            for _ in probe:
                root = os.path.dirname(root)
            return root
    raise FileNotFoundError(
        f'no copy of the NWRD release under {candidates}; attach the Kaggle dataset '
        'bcap52/nwrd-wheat or pass --data-root (scripts/fetch_data.py downloads it)'
    )


def verify_release(root: str, manifest: dict) -> None:
    """Raises RuntimeError unless every file of the manifest is under `root`, byte for byte (its
    size and MD5). Files the manifest does not list are ignored."""
    missing, changed = [], []
    for name, entry in sorted(manifest['files'].items()):
        path = os.path.join(root, *name.split('/'))
        if not os.path.isfile(path):
            missing.append(name)
        elif os.path.getsize(path) != entry['bytes'] or md5_of(path) != entry['md5']:
            changed.append(name)
    if missing or changed:
        raise RuntimeError(
            f'the release under {root} does not match the manifest: {len(missing)} missing '
            f'{missing[:5]}, {len(changed)} changed {changed[:5]}'
        )


@dataclass
class Release:
    root: str
    folders: dict[str, str]  # photograph id -> 'train' or 'test', its folder in the release
    splits: dict[str, list[str]]  # 'train', 'val', 'test' -> photograph ids


def open_release(
    hint: str | None = None, manifest_path: str | None = None, limit: int = 0, log=print
) -> Release:
    """Finds the release, checks every file against the manifest, and splits it. With `limit`,
    each split keeps only its first `limit` photographs (smoke tests)."""
    manifest = load_manifest(manifest_path)
    root = find_release(hint, manifest)
    verify_release(root, manifest)
    log(f'data: {root}: all {len(manifest["files"])} files match the manifest')
    s = splits(manifest)
    if limit:
        s = {k: v[:limit] for k, v in s.items()}
    return Release(root, photo_folders(manifest), s)


def _file(root: str, folder: str, kind: str, pid: str) -> str:
    return os.path.join(root, folder, kind, f'{pid}.{"jpg" if kind == "images" else "png"}')


def read_mask(root: str, folder: str, pid: str) -> np.ndarray:
    """Rust (bool). The release's masks hold only the values 0 and 255."""
    path = _file(root, folder, 'masks', pid)
    grey = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if grey is None:
        raise FileNotFoundError(path)
    return grey > 127


def read_photo(root: str, folder: str, pid: str) -> tuple[np.ndarray, np.ndarray]:
    """One photograph (RGB uint8) and its rust mask (bool), of the same shape.

    The stored pixels are used as they are: every photograph has EXIF orientation 1 or none, and
    every mask matches its photograph's stored size, except photo 101's, which is in no split
    (EXCLUDED_MISALIGNED). A mask of another size is an error: where it belongs on the photograph
    cannot be assumed."""
    path = _file(root, folder, 'images', pid)
    bgr = cv2.imread(path, cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
    if bgr is None:
        raise FileNotFoundError(path)
    rust = read_mask(root, folder, pid)
    (h, w), (mh, mw) = bgr.shape[:2], rust.shape
    if (mh, mw) != (h, w):
        raise ValueError(f'photo {pid}: mask {mw}x{mh} does not match the photograph {w}x{h}')
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), rust


def camera(root: str, folder: str, pid: str) -> str:
    """The camera model in the photograph's EXIF, or 'no EXIF'."""
    from PIL import Image

    with Image.open(_file(root, folder, 'images', pid)) as img:
        model = img.getexif().get(272)  # Model
    model = model.strip('\x00 ') if isinstance(model, str) else ''
    return model or 'no EXIF'


def train_transform():
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(p=0.5),
            A.ElasticTransform(p=0.5),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ]
    )


class RustDataset(torch.utils.data.Dataset):
    """(image tensor, mask tensor of shape (1, H, W), index) for each cut window."""

    def __init__(self, img_dir: str, mask_dir: str, files, transform):
        self.img_dir, self.mask_dir = img_dir, mask_dir
        self.files, self.transform = list(files), transform

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int):
        name = self.files[idx]
        img = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, name)), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(os.path.join(self.mask_dir, name), cv2.IMREAD_GRAYSCALE) > 127
        out = self.transform(image=img, mask=mask.astype(np.float32))
        return out['image'], out['mask'].unsqueeze(0), idx


In [ ]:
%%writefile code/nwrd_public/windows.py
"""Windows of a fixed size on a stride fitted to each photograph, and the training windows' cut.

The release's photographs have 10 sizes, and none has both sides a whole number of 512 px. On a
side of L px, ceil(L / P) windows of P px are spread evenly from 0 to L - P. No padding is
needed, every pixel is covered, and neighbours overlap only by what L lacks of a whole number of
windows (on the photographs used, at most 46 px at 512px and 11 px at 128px). For scoring, each
overlap is split at its midpoint, so every pixel is scored exactly once, by the window whose
centre is nearer (docs/protocol.md §3).

At the NWRD paper's scale a photograph is first resized by 0.5 with an anti-aliased Lanczos filter,
and its mask by area (a half-size pixel is rust when at least half of the area it replaces is).
"""

import json
import math
import os
import random
import time
from concurrent.futures import ProcessPoolExecutor
from itertools import pairwise

import cv2
import numpy as np
from PIL import Image

from .dataset import Release, read_mask, read_photo

# How photographs are resized; recorded with every cut, so a cut made another way is not reused.
PHOTO_RESIZE = 'Pillow Lanczos'


def axis_starts(length: int, size: int) -> list[int]:
    """Start positions of the windows along one side of `length` px."""
    if length < size:
        raise ValueError(
            f'a side of {length} px is shorter than a {size} px window; windows are never padded'
        )
    n = math.ceil(length / size)
    if n == 1:
        return [0]
    return [round(i * (length - size) / (n - 1)) for i in range(n)]


def grid(height: int, width: int, size: int) -> list[tuple[int, int]]:
    """Top-left corners (y, x) of a photograph's windows, row by row."""
    return [(y, x) for y in axis_starts(height, size) for x in axis_starts(width, size)]


def owned_spans(starts: list[int], size: int, length: int) -> list[tuple[int, int]]:
    """For each window along one side, the pixels [lo, hi) it scores: each overlap is split at its
    midpoint, so every pixel belongs to exactly one window, the one whose centre is nearer."""
    bounds = [0] + [(a + b + size) // 2 for a, b in pairwise(starts)] + [length]
    return list(pairwise(bounds))


def scaled_shape(height: int, width: int, scale: float) -> tuple[int, int]:
    return round(height * scale), round(width * scale)


def resize_photo(rgb: np.ndarray, scale: float) -> np.ndarray:
    """The photograph resized by `scale` with a Lanczos filter widened by the reduction, so it
    removes what half the resolution cannot hold: the anti-aliasing the NWRD paper describes.
    (OpenCV's INTER_LANCZOS4 keeps a fixed 8-tap kernel when shrinking and lets fine texture fold
    back into false patterns.)"""
    if scale == 1:
        return rgb
    h, w = scaled_shape(*rgb.shape[:2], scale)
    return np.array(Image.fromarray(rgb).resize((w, h), Image.Resampling.LANCZOS))


def resize_mask(rust: np.ndarray, scale: float) -> np.ndarray:
    """The mask resized by `scale`: a pixel is rust when at least half of its area was rust."""
    if scale == 1:
        return rust
    h, w = scaled_shape(*rust.shape, scale)
    return cv2.resize(rust.astype(np.float32), (w, h), interpolation=cv2.INTER_AREA) >= 0.5


def window_rust(rust: np.ndarray, size: int) -> tuple[list[tuple[int, int]], np.ndarray]:
    """The windows of a mask and the number of rust pixels in each, from its integral image."""
    h, w = rust.shape
    integral = np.zeros((h + 1, w + 1), np.int64)
    np.cumsum(np.cumsum(rust, 0, dtype=np.int64), 1, out=integral[1:, 1:])
    positions = grid(h, w, size)
    ys = np.array([y for y, _ in positions])
    xs = np.array([x for _, x in positions])
    counts = (
        integral[ys + size, xs + size]
        - integral[ys, xs + size]
        - integral[ys + size, xs]
        + integral[ys, xs]
    )
    return positions, counts


def paper_grid(height: int, width: int, size: int = 128) -> list[tuple[int, int]]:
    """The NWRD paper's test patches of a (half-size) photograph, row by row: a grid from the
    origin with stride `size`; what does not fill a whole patch at the right and bottom is not
    scored."""
    return [(r * size, c * size) for r in range(height // size) for c in range(width // size)]


def window_name(pid: str, y: int, x: int) -> str:
    return f'{pid}_y{y:04d}_x{x:04d}.png'


def select_windows(stats: dict, bg_keep_ratio: float, seed: int) -> tuple[list[str], dict]:
    """All windows with rust plus a seeded random fraction of the background-only ones.

    The seed is the *data* seed, fixed across models so every model sees the same windows."""
    rust = sorted(f for f, s in stats.items() if s['pos'] > 0)
    background = sorted(f for f, s in stats.items() if s['pos'] == 0)
    kept = random.Random(seed).sample(background, int(len(background) * bg_keep_ratio))
    info = {'rust_windows': len(rust), 'bg_kept': len(kept), 'bg_total': len(background)}
    return sorted(rust + kept), info


def pos_weight(stats: dict, files: list[str]) -> float:
    """Background-to-rust pixel ratio over `files`: the BCE positive-class weight."""
    pos = sum(stats[f]['pos'] for f in files)
    total = sum(stats[f]['total'] for f in files)
    return (total - pos) / max(pos, 1)


def _count_rust(job) -> list[tuple[str, str, int, int, int, int]]:
    """(photograph, name, y, x, rust pixels, pixels) of every window of one photograph, counted
    on its mask."""
    root, folder, pid, size, scale = job
    positions, counts = window_rust(resize_mask(read_mask(root, folder, pid), scale), size)
    return [
        (pid, window_name(pid, y, x), y, x, int(c), size * size)
        for (y, x), c in zip(positions, counts, strict=True)
    ]


def _cut_photo(job) -> int:
    """Writes the selected windows of one photograph as lossless PNG crops."""
    root, folder, pid, size, scale, windows, dst = job
    rgb, rust = read_photo(root, folder, pid)
    rgb, rust = resize_photo(rgb, scale), resize_mask(rust, scale)
    for name, y, x in windows:
        for path, arr in (
            (
                os.path.join(dst, 'images', name),
                cv2.cvtColor(rgb[y : y + size, x : x + size], cv2.COLOR_RGB2BGR),
            ),
            (
                os.path.join(dst, 'masks', name),
                rust[y : y + size, x : x + size].astype(np.uint8) * 255,
            ),
        ):
            # OpenCV reports a failed write (a full disk, a path it cannot encode) by returning
            # False, not by raising; a cut with a missing window must not be cached as complete.
            if not cv2.imwrite(path, arr):
                raise OSError(f'could not write {path}')
    return len(windows)


def _map(fn, jobs: list, workers: int) -> list:
    if workers <= 1 or len(jobs) < 2:
        return [fn(j) for j in jobs]
    with ProcessPoolExecutor(min(workers, len(jobs))) as ex:
        return list(ex.map(fn, jobs))


def cut_training_windows(
    release: Release,
    ids: list[str],
    dst: str,
    size: int,
    scale: float,
    bg_keep_ratio: float,
    seed: int,
    workers: int = 1,
    log=print,
) -> dict:
    """Cuts the training sample of photographs `ids` into <dst>/{images,masks}/ and returns its
    description, also written to <dst>/windows.json once the cut is complete, with the key
    'dir' (= dst) added.

    Every window of every photograph is counted from the masks; the sample is every window with
    rust plus a `bg_keep_ratio` share of the others (`select_windows`); only the sample is cut.
    Cutting is deterministic, so a finished cut with the same settings is reused, and one with
    other settings is refused rather than overwritten."""
    params = {
        'source': os.path.abspath(release.root),
        'photos': list(ids),
        'window': size,
        'scale': scale,
        'photo_resize': PHOTO_RESIZE,
        'bg_keep_ratio': bg_keep_ratio,
        'seed': seed,
    }
    marker = os.path.join(dst, 'windows.json')
    if os.path.exists(marker):
        with open(marker, encoding='utf-8') as f:
            done = json.load(f)
        if done['params'] != params:
            raise RuntimeError(f'{dst} holds windows cut with other settings; use another dir')
        log(f'windows: reusing {dst}')
        return {**done, 'dir': dst}

    t0 = time.time()
    jobs = [(release.root, release.folders[p], p, size, scale) for p in ids]
    rows = [r for photo in _map(_count_rust, jobs, workers) for r in photo]
    stats = {name: {'pos': pos, 'total': total} for _, name, _, _, pos, total in rows}
    files, sel = select_windows(stats, bg_keep_ratio, seed)
    chosen = set(files)
    by_photo = {}
    for pid, name, y, x, _, _ in rows:
        if name in chosen:
            by_photo.setdefault(pid, []).append((name, y, x))
    for kind in ('images', 'masks'):
        os.makedirs(os.path.join(dst, kind), exist_ok=True)
    cut = [(release.root, release.folders[p], p, size, scale, w, dst) for p, w in by_photo.items()]
    _map(_cut_photo, cut, workers)

    pos = sum(stats[f]['pos'] for f in files)
    info = {
        'params': params,
        'photos': len(ids),
        'windows': len(stats),
        **sel,
        'sample': len(files),
        'rust_pixel_fraction': pos / sum(stats[f]['total'] for f in files),
        'pos_weight': pos_weight(stats, files),
        'seconds': round(time.time() - t0, 1),
        'files': files,
    }
    # Written last and atomically: its presence means every window above was written.
    with open(marker + '.tmp', 'w', encoding='utf-8') as f:
        json.dump(info, f, indent=1)
    os.replace(marker + '.tmp', marker)
    log(
        f'windows: {len(ids)} photographs -> {len(stats)} windows of {size}px at {scale:g}x, '
        f'{sel["rust_windows"]} with rust; sample {len(files)} (pos_weight '
        f'{info["pos_weight"]:.1f}), cut in {info["seconds"]:.0f}s to {dst}'
    )
    return {**info, 'dir': dst}


In [ ]:
%%writefile code/nwrd_public/models.py
"""Model definitions: CANet-B4 (ours), the NWRD paper's UNet, the DeepLabV3+ baselines, and the
512px ablation's variants (docs/protocol.md §9).

Module names inside CANet match the checkpoints trained since the sprint-2 notebook, so every
existing checkpoint loads unchanged.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

from .config import Config

MODELS = {
    'canet_b4': 'CANet, EfficientNet-B4 encoder (ours)',
    'deeplabv3plus_r50': 'DeepLabV3+, ResNet-50 encoder (baseline)',
    'deeplabv3plus_b4': 'DeepLabV3+, EfficientNet-B4 encoder (same-encoder ablation)',
    'deeplabv3plus_r50_scratch': (
        'DeepLabV3+, ResNet-50 encoder, from scratch (pretraining ablation)'
    ),
    'unet_r50': 'UNet decoder on an ImageNet ResNet-50 encoder (decoder ablation)',
    'unet': 'UNet, 4 encoder / 4 decoder blocks, from scratch (NWRD paper baseline)',
    'unet_tuned': 'UNet (paper architecture) with a from-scratch lr (config.unet_tuned_lr)',
}


class ConvBNReLU(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, p: int = 1, d: int = 1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, padding=p, dilation=d, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ContextFlow(nn.Module):
    """One CAM information flow: a shallow encoder-decoder at a given downsample scale."""

    def __init__(self, in_ch: int, out_ch: int, scale: int):
        super().__init__()
        self.scale = scale
        self.encode = ConvBNReLU(in_ch, out_ch)
        self.decode = ConvBNReLU(out_ch, out_ch)

    def forward(self, x):
        h, w = x.shape[-2:]
        # A window larger than the map would pool it to nothing: a 128px input gives a 4x4 map at
        # 1/32, which scale 8 cannot pool. Capped, that flow pools globally. Maps of 8x8 and up
        # (inputs of 256px and up) are untouched.
        k = min(self.scale, h, w)
        xd = F.avg_pool2d(x, k) if k > 1 else x
        f = self.decode(self.encode(xd))
        return F.interpolate(f, (h, w), mode='bilinear', align_corners=False) if k > 1 else f


class AttentionFusion(nn.Module):
    """Re-fusion: a per-pixel, per-channel sigmoid gate computed from the fused features."""

    def __init__(self, in_ch: int):
        super().__init__()
        self.att = nn.Sequential(
            nn.Conv2d(in_ch, in_ch // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch // 4, in_ch, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.att(x)


class CAM(nn.Module):
    """Context aggregation module: a serial dilated global flow plus parallel context
    flows at scales 2/4/8, pre-fused by concatenation and re-fused by the attention gate."""

    def __init__(self, in_ch: int, out_ch: int = 256):
        super().__init__()
        self.global_flow = nn.Sequential(
            ConvBNReLU(in_ch, out_ch, k=3, p=2, d=2),
            ConvBNReLU(out_ch, out_ch, k=3, p=4, d=4),
            ConvBNReLU(out_ch, out_ch, k=3, p=8, d=8),
        )
        self.context_flows = nn.ModuleList(
            [
                ContextFlow(in_ch, out_ch, 2),
                ContextFlow(in_ch, out_ch, 4),
                ContextFlow(in_ch, out_ch, 8),
            ]
        )
        self.pre_fusion = ConvBNReLU(out_ch * 4, out_ch, k=1, p=0)
        self.re_fusion = AttentionFusion(out_ch)
        self.out_conv = ConvBNReLU(out_ch, out_ch)

    def forward(self, x):
        gf = self.global_flow(x)
        cfs = [cf(x) for cf in self.context_flows]
        fused = self.pre_fusion(torch.cat([gf] + cfs, dim=1))
        return self.out_conv(self.re_fusion(fused))


class AsymmetricDecoder(nn.Module):
    """Fuses upsampled 1/32 context with reduced 1/4 low-level features."""

    def __init__(self, high_ch: int, low_ch: int, out_ch: int = 128):
        super().__init__()
        self.low_reduce = ConvBNReLU(low_ch, 48, k=1, p=0)
        self.fuse = nn.Sequential(ConvBNReLU(high_ch + 48, out_ch), ConvBNReLU(out_ch, out_ch))

    def forward(self, high, low):
        high_up = F.interpolate(high, low.shape[-2:], mode='bilinear', align_corners=False)
        return self.fuse(torch.cat([high_up, self.low_reduce(low)], dim=1))


class CANet(nn.Module):
    """CANet with an EfficientNet-B4 encoder: CAM on the 1/32 features, fused with the 1/4
    features by an asymmetric decoder, then a 1/4-resolution head upsampled x4."""

    has_pretrained_encoder = True  # its encoder is frozen for the first epochs (train.py)

    def __init__(self, dropout: float = Config.dropout, pretrained: bool = True):
        super().__init__()
        from efficientnet_pytorch import EfficientNet

        self.encoder = (
            EfficientNet.from_pretrained('efficientnet-b4')
            if pretrained
            else EfficientNet.from_name('efficientnet-b4')
        )
        self.reduce = ConvBNReLU(1792, 512, k=1, p=0)
        self.cam = CAM(512, 256)
        self.decoder = AsymmetricDecoder(256, 32, 128)  # reduction_2: 32ch at 1/4
        self.dropout = nn.Dropout2d(dropout)
        self.head = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(64, 1, 1)
        )

    def forward(self, x):
        # One encoder pass yields both the 1/32 features (reduction_6, which equals
        # extract_features(x)) and the 1/4 features (reduction_2).
        ep = self.encoder.extract_endpoints(x)
        enc, low = ep['reduction_6'], ep['reduction_2']
        x_cam = self.cam(self.reduce(enc))
        x_dec = self.dropout(self.decoder(x_cam, low))
        return F.interpolate(self.head(x_dec), scale_factor=4, mode='bilinear', align_corners=False)


class _DoubleConv(nn.Sequential):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class UNet(nn.Module):
    """Plain UNet as used by the NWRD paper (Anwar et al., Sensors 2023): 4 encoder and 4
    decoder blocks, 64-1024 channels, transposed-conv upsampling, no pretraining."""

    has_pretrained_encoder = False  # trained from scratch, so never frozen

    def __init__(self, base: int = 64):
        super().__init__()
        ch = [base * 2**i for i in range(5)]
        self.inc = _DoubleConv(3, ch[0])
        self.encoder = nn.ModuleList(_DoubleConv(ch[i], ch[i + 1]) for i in range(4))
        self.up = nn.ModuleList(
            nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(4))
        )
        self.decoder = nn.ModuleList(_DoubleConv(ch[i] * 2, ch[i]) for i in reversed(range(4)))
        self.head = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x):
        skips = [self.inc(x)]
        for down in self.encoder:
            skips.append(down(F.max_pool2d(skips[-1], 2)))
        x = skips.pop()
        for up, dec in zip(self.up, self.decoder, strict=True):
            x = dec(torch.cat([skips.pop(), up(x)], 1))
        return self.head(x)


def build_model(name: str, pretrained: bool = True, dropout: float = Config.dropout) -> nn.Module:
    """All models return raw logits of shape (B, 1, H, W) and expose `.encoder`.

    A model with `has_pretrained_encoder = False` is never frozen (train.py). An smp model has
    it set to whether ImageNet weights were loaded, so `deeplabv3plus_r50_scratch`, whose
    encoder is always random, is never frozen."""
    if name == 'canet_b4':
        return CANet(dropout=dropout, pretrained=pretrained)
    if name in ('unet', 'unet_tuned'):
        return UNet()
    if name in ('deeplabv3plus_r50', 'deeplabv3plus_b4', 'deeplabv3plus_r50_scratch', 'unet_r50'):
        import segmentation_models_pytorch as smp

        weights = 'imagenet' if pretrained and not name.endswith('_scratch') else None
        model = (smp.Unet if name == 'unet_r50' else smp.DeepLabV3Plus)(
            encoder_name='efficientnet-b4' if name.endswith('_b4') else 'resnet50',
            encoder_weights=weights,
            classes=1,
        )
        model.has_pretrained_encoder = weights is not None
        return model
    raise ValueError(f'unknown model {name!r}; choose from {sorted(MODELS)}')


def load_checkpoint(model: nn.Module, path: str, map_location='cpu') -> dict:
    """Loads a checkpoint written by the pipeline ({'model': state_dict, ...}) into `model`
    and returns the whole checkpoint. Only tensors and plain data are unpickled."""
    ckpt = torch.load(path, map_location=map_location, weights_only=True)
    model.load_state_dict(ckpt['model'])
    return ckpt


In [ ]:
%%writefile code/nwrd_public/metrics.py
"""Pixel metrics pooled over a whole split.

The sprint notebooks averaged IoU per batch, which scores a correctly predicted all-background
batch as 0 and depresses the curve (~0.58 vs ~0.81 pooled). Everything here pools the confusion
counts over every pixel of the split, then computes the ratios once.

Confusion counts are always ordered (tp, fp, fn, tn).
"""

import numpy as np
import torch

EPS = 1e-12


def scores(tp, fp, fn, tn) -> dict[str, float]:
    tp, fp, fn, tn = (float(v) for v in (tp, fp, fn, tn))
    return {
        'IoU': tp / (tp + fp + fn + EPS),
        'Dice': 2 * tp / (2 * tp + fp + fn + EPS),
        'Precision': tp / (tp + fp + EPS),
        'Recall': tp / (tp + fn + EPS),
        'Specificity': tn / (tn + fp + EPS),
        'Accuracy': (tp + tn) / (tp + tn + fp + fn + EPS),
    }


def iou(counts: np.ndarray) -> np.ndarray:
    """IoU of every row of an array of (..., 4) confusion counts."""
    tp, fp, fn = counts[..., 0], counts[..., 1], counts[..., 2]
    return tp / np.maximum(tp + fp + fn, 1)


def dice(counts: np.ndarray) -> np.ndarray:
    """Dice of every row of an array of (..., 4) confusion counts."""
    tp, fp, fn = counts[..., 0], counts[..., 1], counts[..., 2]
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)


def per_image_counts(probs: torch.Tensor, target: torch.Tensor, threshold: float) -> np.ndarray:
    """(B, 4) int64 array of tp, fp, fn, tn per image. Decision rule: prob >= threshold."""
    pred = probs >= threshold
    tgt = target > 0.5
    dims = tuple(range(1, probs.dim()))
    tp = (pred & tgt).sum(dims)
    fp = (pred & ~tgt).sum(dims)
    fn = (~pred & tgt).sum(dims)
    tn = (~pred & ~tgt).sum(dims)
    return torch.stack([tp, fp, fn, tn], 1).cpu().numpy().astype(np.int64)


class ProbHistogram:
    """Histograms of predicted probability for rust and background pixels.

    Gives exact pooled confusion counts at any threshold on a 1/bins grid without keeping
    every prediction in memory."""

    def __init__(self, bins: int = 1000):
        self.bins = bins
        self.pos = np.zeros(bins, np.int64)
        self.neg = np.zeros(bins, np.int64)

    def update(self, probs: torch.Tensor, target: torch.Tensor) -> None:
        q = (probs.float() * self.bins).long().clamp_(0, self.bins - 1).flatten()
        t = (target > 0.5).flatten()
        self.pos += torch.bincount(q[t], minlength=self.bins).cpu().numpy()
        self.neg += torch.bincount(q[~t], minlength=self.bins).cpu().numpy()

    def counts(self, threshold: float) -> tuple[int, int, int, int]:
        k = int(round(threshold * self.bins))
        tp, fp = self.pos[k:].sum(), self.neg[k:].sum()
        return tp, fp, self.pos.sum() - tp, self.neg.sum() - fp

    def sweep(self, lo: float = 0.05, hi: float = 0.95, step: float = 0.01) -> list[dict]:
        rows = []
        for t in np.round(np.arange(lo, hi + step / 2, step), 4):
            s = scores(*self.counts(t))
            rows.append({'threshold': float(t), **s})
        return rows

    def best_threshold(self, **kw) -> tuple[float, list[dict]]:
        """IoU = Dice / (2 - Dice), so maximising either picks the same threshold."""
        rows = self.sweep(**kw)
        return max(rows, key=lambda r: r['IoU'])['threshold'], rows


In [ ]:
%%writefile code/nwrd_public/train.py
"""Training for one (model, seed) run. Resumable from last.pt."""

import json
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn

from .config import Config
from .dataset import Release, RustDataset, train_transform
from .evaluate import load_photos, score
from .metrics import scores
from .models import build_model


class CombinedLoss(nn.Module):
    """dice_w * Dice loss + bce_w * BCE, both on logits; BCE weights rust pixels by pos_weight."""

    def __init__(self, dice_w: float, bce_w: float, pos_weight: torch.Tensor):
        super().__init__()
        import segmentation_models_pytorch as smp

        self.dice_w, self.bce_w = dice_w, bce_w
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, target):
        return self.dice_w * self.dice(logits, target) + self.bce_w * self.bce(logits, target)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _seed_worker(worker_id: int) -> None:
    """Seeds a DataLoader worker's augmentation from the seed PyTorch derives for that worker
    from the loader's generator, so augmentation depends on (seed, epoch, worker) only.
    Albumentations 2 seeds its transforms from the OS otherwise, not from the global seed."""
    info = torch.utils.data.get_worker_info()
    info.dataset.transform.set_random_seed(info.seed % 2**32)


def make_train_loader(cfg: Config, windows: dict):
    """The loader over the cut training windows, and its generator (reseed it every epoch)."""
    g = torch.Generator()
    loader = torch.utils.data.DataLoader(
        RustDataset(
            os.path.join(windows['dir'], 'images'),
            os.path.join(windows['dir'], 'masks'),
            windows['files'],
            train_transform(),
        ),
        batch_size=cfg.batch_size,
        shuffle=True,
        drop_last=True,
        generator=g,
        worker_init_fn=_seed_worker,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    return loader, g


def train_run(
    cfg: Config,
    release: Release,
    windows: dict,
    seed: int,
    run_dir: str,
    log=print,
    session_start=None,
) -> bool:
    """Trains one run to cfg.epochs on the cut windows, keeping the checkpoint with the best
    pooled IoU at threshold 0.5 over the whole validation photographs as best.pt. Returns False
    if the time budget stopped it (resume later)."""
    os.makedirs(run_dir, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    amp = cfg.amp and device.type == 'cuda'
    torch.backends.cudnn.benchmark = True
    set_seed(seed)

    train_loader, gen = make_train_loader(cfg, windows)
    val_photos = load_photos(release, release.splits['val'], cfg.scale)
    keys = ('windows', 'rust_windows', 'bg_kept', 'bg_total', 'sample', 'pos_weight')
    info = {**{k: windows[k] for k in keys}, 'val_photos': len(val_photos)}
    log(f'[{cfg.model} s{seed}] data: {info}')
    model = build_model(cfg.model, pretrained=cfg.pretrained, dropout=cfg.dropout).to(device)
    criterion = CombinedLoss(
        cfg.dice_weight, cfg.bce_weight, torch.tensor([windows['pos_weight']], device=device)
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=cfg.sched_t0, T_mult=cfg.sched_tmult, eta_min=cfg.sched_eta_min
    )
    scaler = torch.amp.GradScaler(device.type, enabled=amp)

    history = {
        k: []
        for k in (
            'train_loss',
            'val_loss',
            'val_iou',
            'val_dice',
            'val_iou_best_thr',
            'lr',
            'epoch_seconds',
        )
    }
    start_epoch, best_iou = 0, -1.0
    last_path, best_path = os.path.join(run_dir, 'last.pt'), os.path.join(run_dir, 'best.pt')
    if os.path.exists(last_path):
        # Load on the CPU: torch.set_rng_state only accepts a CPU tensor, and load_state_dict
        # moves model and optimizer state onto the model's device by itself.
        ck = torch.load(last_path, map_location='cpu', weights_only=True)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        scaler.load_state_dict(ck['scaler'])
        torch.set_rng_state(ck['rng_cpu'])
        if device.type == 'cuda' and ck.get('rng_cuda') is not None:
            torch.cuda.set_rng_state_all(ck['rng_cuda'])
        start_epoch, best_iou, history = ck['epoch'], ck['best_iou'], ck['history']
        log(f'[{cfg.model} s{seed}] resumed at epoch {start_epoch}')

    t_start = time.time()
    first_in_session = session_start is None
    session_start = t_start if first_in_session else session_start
    for epoch in range(start_epoch, cfg.epochs):
        # The budget covers the whole session, not this run. The session's first run always
        # gets one epoch, so every session makes progress; the headroom between the budget
        # and the platform limit covers that epoch when no per-epoch estimate exists yet.
        if cfg.time_budget_hours and (epoch > start_epoch or not first_in_session):
            done = epoch - start_epoch
            per_epoch = (time.time() - t_start) / done if done else 0.0
            if time.time() - session_start + 1.5 * per_epoch > cfg.time_budget_hours * 3600:
                log(f'[{cfg.model} s{seed}] time budget reached at epoch {epoch}; resume later')
                return False

        # Freezing only makes sense for an ImageNet-pretrained encoder.
        frozen = epoch < cfg.freeze_encoder_epochs and getattr(
            model, 'has_pretrained_encoder', True
        )
        for p in model.encoder.parameters():
            p.requires_grad = not frozen

        # Shuffle order and augmentation depend on (seed, epoch) only, so a run is reproducible
        # from its seed and a resumed run sees the batches it would have seen uninterrupted.
        # Worker processes reseed their augmentation from `gen` (_seed_worker); without
        # workers, the augmentation runs here.
        gen.manual_seed(seed * 10_000 + epoch)
        train_loader.dataset.transform.set_random_seed(seed * 10_000 + epoch)
        t0 = time.time()
        model.train()
        loss_sum, n = 0.0, 0
        for imgs, masks, _ in train_loader:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, enabled=amp):
                logits = model(imgs)
            loss = criterion(logits.float(), masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * imgs.size(0)
            n += imgs.size(0)
        lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        # Validation scores whole photographs at native resolution, as the final evaluation does.
        v = score(model, val_photos, cfg, device, criterion=criterion)
        val = scores(*v.hist.counts(0.5))
        _, sweep = v.hist.best_threshold(lo=cfg.thr_lo, hi=cfg.thr_hi, step=cfg.thr_step)
        for k, value in (
            ('train_loss', loss_sum / n),
            ('val_loss', v.loss),
            ('val_iou', val['IoU']),
            ('val_dice', val['Dice']),
            ('val_iou_best_thr', max(r['IoU'] for r in sweep)),
            ('lr', lr),
            ('epoch_seconds', time.time() - t0),
        ):
            history[k].append(value)
        log(
            f'[{cfg.model} s{seed}] ep {epoch + 1:02d}/{cfg.epochs} '
            f'{"(enc frozen) " if frozen else ""}train {loss_sum / n:.4f} val {v.loss:.4f} '
            f'IoU@0.5 {val["IoU"]:.4f} Dice@0.5 {val["Dice"]:.4f} '
            f'({history["epoch_seconds"][-1]:.0f}s)'
        )

        # Model selection: pooled val IoU at the fixed 0.5 threshold.
        if val['IoU'] > best_iou:
            best_iou = val['IoU']
            torch.save(
                {
                    'model': model.state_dict(),
                    'epoch': epoch + 1,
                    'val_iou@0.5': best_iou,
                    'model_name': cfg.model,
                    'seed': seed,
                },
                best_path,
            )
        torch.save(
            {
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'scaler': scaler.state_dict(),
                'rng_cpu': torch.get_rng_state(),
                'rng_cuda': torch.cuda.get_rng_state_all() if device.type == 'cuda' else None,
                'epoch': epoch + 1,
                'best_iou': best_iou,
                'history': history,
            },
            last_path,
        )
        with open(os.path.join(run_dir, 'history.json'), 'w') as f:
            json.dump({'history': history, 'data': info, 'best_val_iou@0.5': best_iou}, f, indent=1)

    os.remove(last_path)  # optimizer state is ~2/3 of the file; best.pt is what we keep
    with open(os.path.join(run_dir, 'TRAINED'), 'w') as f:
        f.write(f'best val IoU@0.5 {best_iou:.6f}\n')
    return True


In [ ]:
%%writefile code/nwrd_public/evaluate.py
"""Scoring a trained run on whole photographs.

Each validation and test photograph is predicted whole: its windows (nwrd_public/windows.py) go
through the model, and each pixel takes the probability of the window that owns it. At the
paper's scale the half-size map is resized bilinearly to the native size. Every pixel is then
scored once, at native resolution, against the full-size mask, so no padding is scored, no edge is
dropped, and both scales are scored on the same pixels.

Protocol: the binarisation threshold is chosen on the validation photographs, frozen, and applied
once to the test photographs. Test metrics at 0.5 are reported too.

Each model is also scored on exactly the NWRD paper's 2,737 test patches: 128px patches of the
half-size test photographs on a grid from the origin, in the order the authors' test.py read
them. They are scored pooled, on rust-positive patches only (the paper's Table 4 setting), and
averaged per batch of 512 as test.py did (docs/protocol.md §3).
"""

import json
import os
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn.functional as F

from .config import Config
from .dataset import MEAN, STD, Release, camera, read_photo
from .metrics import ProbHistogram, per_image_counts, scores
from .models import build_model, load_checkpoint
from .windows import axis_starts, owned_spans, paper_grid, resize_mask, resize_photo, scaled_shape

PAPER_PATCH = 128  # the paper's patch size, on photographs halved in size
PAPER_BATCH = 512  # the batch size of the authors' test.py


@dataclass
class EvalPhoto:
    id: str
    image: np.ndarray  # RGB uint8 at the working scale
    mask: np.ndarray  # rust (bool) at native resolution: what is scored
    mask_work: np.ndarray  # rust (bool) at the working scale: the windows' loss targets
    camera: str


def load_photos(release: Release, ids: list[str], scale: float) -> list[EvalPhoto]:
    photos = []
    for pid in ids:
        folder = release.folders[pid]
        rgb, rust = read_photo(release.root, folder, pid)
        photos.append(
            EvalPhoto(
                pid,
                resize_photo(rgb, scale),
                rust,
                resize_mask(rust, scale),
                camera(release.root, folder, pid),
            )
        )
    return photos


def predict(
    model, photo: EvalPhoto, size: int, device, amp: bool, batch_size: int, criterion=None
) -> tuple[torch.Tensor, float, int]:
    """The photograph's rust probability at the working scale, each pixel from the window that
    owns it; the loss summed over its windows (0 without a criterion); the number of windows."""
    h, w = photo.mask_work.shape
    ys, xs = axis_starts(h, size), axis_starts(w, size)
    windows = [
        (y, x, rows, cols)
        for y, rows in zip(ys, owned_spans(ys, size, h), strict=True)
        for x, cols in zip(xs, owned_spans(xs, size, w), strict=True)
    ]
    img = torch.from_numpy(photo.image).to(device).permute(2, 0, 1)
    mean = torch.tensor(MEAN, device=device).view(3, 1, 1) * 255
    std = torch.tensor(STD, device=device).view(3, 1, 1) * 255
    target = torch.from_numpy(photo.mask_work).to(device) if criterion is not None else None
    prob = torch.empty((h, w), device=device)
    loss_sum = 0.0
    model.eval()
    with torch.no_grad():
        for i in range(0, len(windows), batch_size):
            batch = windows[i : i + batch_size]
            inputs = torch.stack([img[:, y : y + size, x : x + size] for y, x, _, _ in batch])
            with torch.autocast(device.type, enabled=amp):
                logits = model((inputs.float() - mean) / std)
            logits = logits.float()
            if criterion is not None:
                t = torch.stack([target[y : y + size, x : x + size] for y, x, _, _ in batch])
                loss_sum += criterion(logits, t.unsqueeze(1).float()).item() * len(batch)
            p = torch.sigmoid(logits[:, 0])
            for k, (y, x, (r0, r1), (c0, c1)) in enumerate(batch):
                prob[r0:r1, c0:c1] = p[k, r0 - y : r1 - y, c0 - x : c1 - x]
    return prob, loss_sum, len(windows)


def to_native(prob: torch.Tensor, shape: tuple[int, int]) -> torch.Tensor:
    """A working-scale probability map resized bilinearly to the native size."""
    if tuple(prob.shape) == tuple(shape):
        return prob
    return F.interpolate(prob[None, None], size=shape, mode='bilinear', align_corners=False)[0, 0]


@dataclass
class Scores:
    hist: ProbHistogram  # every native pixel of every photograph
    loss: float  # mean loss per window (nan without a criterion)
    photo_counts: dict[float, np.ndarray]  # threshold -> (photographs, 4) confusion counts
    paper_counts: dict[float, np.ndarray]  # threshold -> (paper patches, 4)
    paper_index: list[tuple[str, int, int]]  # (photograph, row, col) of each paper patch
    paper_rust: np.ndarray  # whether each paper patch has rust


def _paper_patches(prob_half: torch.Tensor, rust_half: torch.Tensor):
    """Positions and stacked (N, 1, 128, 128) probabilities and masks of the paper's patches."""
    pos = paper_grid(*prob_half.shape, PAPER_PATCH)
    s = PAPER_PATCH
    p = torch.stack([prob_half[y : y + s, x : x + s] for y, x in pos]).unsqueeze(1)
    m = torch.stack([rust_half[y : y + s, x : x + s] for y, x in pos]).unsqueeze(1)
    return pos, p, m


def score(
    model, photos: list[EvalPhoto], cfg: Config, device, criterion=None, thresholds=(), paper=False
) -> Scores:
    """Predicts each photograph whole and scores it at native resolution. Per-photograph counts
    are kept at each of `thresholds`; with `paper`, so are those of the paper's test patches, in
    the order of `photos`."""
    amp = cfg.amp and device.type == 'cuda'
    hist = ProbHistogram()
    photo_counts = {t: [] for t in thresholds}
    paper_counts = {t: [] for t in thresholds}
    paper_index, paper_rust = [], []
    loss_sum, n_windows = 0.0, 0
    for photo in photos:
        prob, loss, n = predict(
            model, photo, cfg.window, device, amp, cfg.batch_size, criterion=criterion
        )
        loss_sum, n_windows = loss_sum + loss, n_windows + n
        mask = torch.from_numpy(photo.mask).to(device)
        native = to_native(prob, photo.mask.shape)
        hist.update(native, mask)
        for t in thresholds:
            photo_counts[t].append(per_image_counts(native[None, None], mask[None, None], t)[0])
        if paper:
            # The paper's half-size photograph: the working scale itself at 0.5, otherwise the
            # native prediction resized by area, as the masks are (windows.resize_mask).
            half = scaled_shape(*photo.mask.shape, 0.5)
            if tuple(prob.shape) == half:
                prob_half, rust_half = prob, photo.mask_work
            else:
                prob_half = F.interpolate(native[None, None], size=half, mode='area')[0, 0]
                rust_half = resize_mask(photo.mask, 0.5)
            pos, p, m = _paper_patches(prob_half, torch.from_numpy(rust_half).to(device))
            paper_index += [(photo.id, y // PAPER_PATCH, x // PAPER_PATCH) for y, x in pos]
            paper_rust.append(m.flatten(1).any(1).cpu().numpy())
            for t in thresholds:
                paper_counts[t].append(per_image_counts(p, m, t))
    return Scores(
        hist=hist,
        loss=loss_sum / n_windows if criterion is not None else float('nan'),
        photo_counts={t: np.stack(v) for t, v in photo_counts.items()},
        paper_counts={t: np.concatenate(v) for t, v in paper_counts.items() if v},
        paper_index=paper_index,
        paper_rust=np.concatenate(paper_rust) if paper_rust else np.zeros(0, bool),
    )


def paper_order(ids: list[str]) -> list[str]:
    """The test photographs in the order the authors' loader read them: file names sorted as
    text (105, 123, 25, ...)."""
    return sorted(ids)


def authors_average(counts: np.ndarray, batch: int = PAPER_BATCH) -> dict[str, float]:
    """Precision, recall and F1 as the authors' test.py computed them: scikit-learn's binary
    scores on each batch of `batch` consecutive patches (a zero division counts as 0), weighted
    by the batch's number of patches."""
    total = {'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0}
    for i in range(0, len(counts), batch):
        part = counts[i : i + batch]
        tp, fp, fn, _ = (float(v) for v in part.sum(0))
        n = len(part)
        total['Precision'] += n * (tp / (tp + fp) if tp + fp else 0.0)
        total['Recall'] += n * (tp / (tp + fn) if tp + fn else 0.0)
        total['F1'] += n * (2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
    return {k: v / len(counts) if len(counts) else 0.0 for k, v in total.items()}


def paper_scores(counts: np.ndarray, rust: np.ndarray, batch: int = PAPER_BATCH) -> dict:
    """The paper's patches scored pooled and with the authors' averaging, over all of them and
    over those with rust only (the paper's Table 4 removed the background-only patches; the
    authors' code does not show how, so those batches are formed after the removal)."""
    return {
        'pooled': scores(*counts.sum(0)),
        'rust_only': scores(*counts[rust].sum(0)),
        'authors': authors_average(counts, batch),
        'authors_rust_only': authors_average(counts[rust], batch),
    }


def evaluate_run(cfg: Config, release: Release, run_dir: str, log=print) -> dict:
    """Picks the threshold on validation, scores test, and writes metrics.json,
    val_threshold_sweep.json, test_counts.npz and paper_patches.npz into `run_dir`."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_model(cfg.model, pretrained=False, dropout=cfg.dropout).to(device)
    ckpt = load_checkpoint(model, os.path.join(run_dir, 'best.pt'), map_location=device)

    val_photos = load_photos(release, release.splits['val'], cfg.scale)
    val = score(model, val_photos, cfg, device)
    thr, val_sweep = val.hist.best_threshold(lo=cfg.thr_lo, hi=cfg.thr_hi, step=cfg.thr_step)
    at_edge = thr in (cfg.thr_lo, cfg.thr_hi)

    test_ids = paper_order(release.splits['test'])
    photos = load_photos(release, test_ids, cfg.scale)
    thresholds = tuple(dict.fromkeys((thr, 0.5)))
    test = score(model, photos, cfg, device, thresholds=thresholds, paper=True)
    pos, neg = test.hist.pos.sum(), test.hist.neg.sum()
    res = {
        'model': cfg.model,
        'window': cfg.window,
        'scale': cfg.scale,
        'best_epoch': ckpt.get('epoch'),
        'threshold': thr,
        'threshold_at_grid_edge': bool(at_edge),
        'val@thr': scores(*val.hist.counts(thr)),
        'val@0.5': scores(*val.hist.counts(0.5)),
        'test@thr': scores(*test.hist.counts(thr)),
        'test@0.5': scores(*test.hist.counts(0.5)),
        'val_photos': len(val_photos),
        'test_photos': len(photos),
        'test_rust_pixel_fraction': float(pos / (pos + neg)),
        'paper_patches': {
            'patches': len(test.paper_index),
            'rust_patches': int(test.paper_rust.sum()),
            '@thr': paper_scores(test.paper_counts[thr], test.paper_rust),
            '@0.5': paper_scores(test.paper_counts[0.5], test.paper_rust),
        },
    }
    with open(os.path.join(run_dir, 'metrics.json'), 'w') as f:
        json.dump(res, f, indent=2)
    with open(os.path.join(run_dir, 'val_threshold_sweep.json'), 'w') as f:
        json.dump(val_sweep, f)
    # Per-photograph counts drive the photograph bootstrap in compare.py; the paper patches'
    # counts let any average of them be recomputed without the model.
    np.savez_compressed(
        os.path.join(run_dir, 'test_counts.npz'),
        photos=np.array(test_ids),
        cameras=np.array([p.camera for p in photos]),
        at_thr=test.photo_counts[thr],
        at_05=test.photo_counts[0.5],
    )
    index = np.array(test.paper_index, dtype=object)
    np.savez_compressed(
        os.path.join(run_dir, 'paper_patches.npz'),
        photo=index[:, 0].astype(str),
        row=index[:, 1].astype(np.int32),
        col=index[:, 2].astype(np.int32),
        rust=test.paper_rust,
        at_thr=test.paper_counts[thr],
        at_05=test.paper_counts[0.5],
    )
    t, pp = res['test@thr'], res['paper_patches']['@0.5']['authors']
    log(
        f'[{cfg.model} {os.path.basename(run_dir)}] thr {thr:.2f}'
        f'{" (GRID EDGE)" if at_edge else ""} | test IoU {t["IoU"]:.4f} Dice {t["Dice"]:.4f} '
        f"P {t['Precision']:.4f} R {t['Recall']:.4f} | paper patches @0.5, authors' "
        f'averaging: P {pp["Precision"]:.4f} R {pp["Recall"]:.4f} F1 {pp["F1"]:.4f}'
    )
    return res


In [ ]:
%%writefile code/nwrd_public/compare.py
"""The side-by-side log of the three models at one scale, and each model's own summary.

Outputs in a scale's folder (results/512px or results/128px): comparison.md, comparison.json,
figures/training_curves.png and <model>/summary.md; and results/summary.csv over both scales.
Every number is computed from the files each run saves (metrics.json, history.json and the
per-photograph counts in test_counts.npz), so the reports are rebuilt without checkpoints or
photographs. Each run's paper_patches.npz keeps the per-patch counts behind its paper-patch
scores, for recomputing other averages offline:

    python -m nwrd_public.compare results
"""

import argparse
import csv
import glob
import json
import os
import sys

import numpy as np

from .config import COMPARED_MODELS
from .metrics import dice, iou
from .models import MODELS
from .windows import grid, scaled_shape

METRICS = ('IoU', 'Dice', 'Precision', 'Recall', 'Specificity', 'Accuracy')
OURS = 'canet_b4'
SCALES = ('512px', '128px')

# Published NWRD paper results (Anwar et al., Sensors 2023, doi:10.3390/s23156942) for UNet + APF
# on the release's 10 test photographs. Table 3: the 2,737 patches the paper's test.py scores
# (128px patches of the half-size photographs), averaged per batch as that code does. Table 4: a
# separately trained UNet (stride 128, background-only patches removed from training and test,
# paper §5), on the test patches with rust; its IoU was not computed by test.py.
PAPER_TEST_PATCHES = 2737
PAPER = [
    {
        'table': 'Table 3',
        'patches': 'all test patches',
        'kind': 'authors',
        'scores': {'Precision': 0.506, 'Recall': 0.624, 'F1': 0.557},
    },
    {
        'table': 'Table 4',
        'patches': 'background-only patches removed',
        'kind': 'authors_rust_only',
        'scores': {'Precision': 0.593, 'Recall': 0.552, 'F1': 0.564, 'IoU': 0.438},
    },
]
PAPER_ROWS = (  # (threshold key, kind, label) of our runs on the paper's patches
    ('@0.5', 'authors', "all patches, the authors' averaging, threshold 0.5"),
    ('@0.5', 'authors_rust_only', "rust patches (≈ Table 4), the authors' averaging, 0.5"),
    ('@thr', 'pooled', 'all patches, pooled, validation threshold'),
    ('@thr', 'rust_only', 'rust patches, pooled, validation threshold'),
)
# The two common sizes of the release, for the time per photograph: (height, width) in pixels.
PHOTO_SIZES = {
    'Nikon D3200, 6016×4000': (4000, 6016),
    'Huawei PAR-LX1M, 4608×3456 or 3456×4608': (3456, 4608),
}

VERDICTS = {
    'improves': 'improves on',
    'comparable': 'is comparable to',
    'worse': 'is worse than',
}


def _read_json(path: str):
    if not os.path.exists(path):
        return None
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def load_runs(scale_dir: str) -> dict:
    """{model: {seed: run}} for every run with a metrics.json. A run holds its directory and
    metrics, plus 'history' (the whole history.json), 'photos', 'cameras' and 'counts' (the
    test photographs' confusion counts at the run's threshold) and 'paper' (the path of its
    paper_patches.npz) where those files exist."""
    runs = {}
    for path in sorted(glob.glob(os.path.join(scale_dir, '*', 'seed*', 'metrics.json'))):
        run_dir = os.path.dirname(path)
        model = os.path.basename(os.path.dirname(run_dir))
        seed = int(os.path.basename(run_dir)[len('seed') :])
        run = {'dir': run_dir, 'metrics': _read_json(path)}
        history = _read_json(os.path.join(run_dir, 'history.json'))
        if history is not None:
            run['history'] = history
        counts = os.path.join(run_dir, 'test_counts.npz')
        if os.path.exists(counts):
            with np.load(counts) as c:
                run['photos'], run['cameras'] = c['photos'], c['cameras']
                run['counts'] = c['at_thr'].astype(np.int64)
        if os.path.exists(os.path.join(run_dir, 'paper_patches.npz')):
            run['paper'] = os.path.join(run_dir, 'paper_patches.npz')
        runs.setdefault(model, {})[seed] = run
    return runs


def paired_bootstrap(
    ours: dict, base: dict, seeds: list, n_boot: int = 5000, rng_seed: int = 0
) -> dict:
    """Paired bootstrap of the seed-averaged difference in pooled test IoU and Dice. Each
    resample draws test photographs with replacement, the same draws for both models and every
    seed: pixels of one photograph share plant, light and annotator, so the photograph is the
    unit of resampling."""
    for s in seeds:
        if not np.array_equal(ours[s]['photos'], base[s]['photos']):
            raise ValueError(f'seed {s}: the two models were tested on different photographs')
    n = len(ours[seeds[0]]['photos'])
    rng = np.random.default_rng(rng_seed)
    draws = rng.integers(0, n, size=(n_boot, n))
    w = np.stack([np.bincount(d, minlength=n) for d in draws])
    out = {}
    for name, fn in (('IoU', iou), ('Dice', dice)):
        d = np.mean([fn(w @ ours[s]['counts']) - fn(w @ base[s]['counts']) for s in seeds], 0)
        point = np.mean(
            [fn(ours[s]['counts'].sum(0)) - fn(base[s]['counts'].sum(0)) for s in seeds]
        )
        out[name] = {
            'delta': float(point),
            'ci95': [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
            'p_two_sided': float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))),
            'units': int(n),
        }
    return out


def seed_ttest(ours: dict, base: dict, seeds: list) -> dict | None:
    """Paired t-test over seeds of test IoU; it captures training-run variance."""
    if len(seeds) < 2:
        return None
    from scipy import stats

    a = [ours[s]['metrics']['test@thr']['IoU'] for s in seeds]
    b = [base[s]['metrics']['test@thr']['IoU'] for s in seeds]
    t, p = stats.ttest_rel(a, b)
    return {'t': float(t), 'p': float(p), 'n_seeds': len(seeds)}


def seed_wins(ours: dict, base: dict, seeds: list) -> int:
    """Number of seeds on which `ours` has the higher test IoU."""
    return sum(
        ours[s]['metrics']['test@thr']['IoU'] > base[s]['metrics']['test@thr']['IoU'] for s in seeds
    )


def verdict(boot_photo_iou: dict, wins: int) -> str:
    """The decision rule, fixed before any result existed (docs/protocol.md §5).

    'improves': higher mean test IoU, the 95% CI of the IoU difference that resamples test
    photographs excludes 0, and CANet wins on at least 2 of 3 seeds. 'worse': that CI lies
    entirely below 0. 'comparable': otherwise. The rule assumes all three seeds; with fewer,
    the reading is interim."""
    lo, hi = boot_photo_iou['ci95']
    if boot_photo_iou['delta'] > 0 and lo > 0 and wins >= 2:
        return 'improves'
    if hi < 0:
        return 'worse'
    return 'comparable'


def significance(runs: dict) -> dict:
    """CANet-B4 against each pre-registered baseline, on the seeds both have. The ablation's
    models (config.ABLATION_MODELS) are exploratory and never judged by the decision rule."""
    tests = {}
    if OURS not in runs:
        return tests
    for base in (m for m in runs if m != OURS and m in COMPARED_MODELS):
        seeds = sorted(set(runs[OURS]) & set(runs[base]))
        if not seeds:
            continue
        boot = paired_bootstrap(runs[OURS], runs[base], seeds)
        wins = seed_wins(runs[OURS], runs[base], seeds)
        tests[base] = {
            'seeds': seeds,
            'bootstrap_photo': boot,
            'seed_ttest': seed_ttest(runs[OURS], runs[base], seeds),
            'canet_wins': wins,
            'verdict': verdict(boot['IoU'], wins),
        }
    return tests


def mean_std(vals) -> tuple[float, float]:
    """Mean and sample standard deviation (0 for a single value)."""
    vals = np.asarray(vals, float)
    return float(vals.mean()), float(vals.std(ddof=1)) if len(vals) > 1 else 0.0


def _fmt(m: float, s: float, n: int) -> str:
    return f'{m:.4f} ± {s:.4f}' if n > 1 else f'{m:.4f}'


def summarise(runs: dict) -> dict:
    summary = {}
    for model, seeds in runs.items():
        ms = [seeds[s]['metrics'] for s in sorted(seeds)]
        hs = [seeds[s]['history']['history'] for s in sorted(seeds)]
        row = {'seeds': sorted(seeds), 'n': len(ms)}
        for split in ('test@thr', 'test@0.5', 'val@thr'):
            row[split] = {k: mean_std([m[split][k] for m in ms]) for k in METRICS}
        row['threshold'] = mean_std([m['threshold'] for m in ms])
        row['threshold_at_grid_edge'] = any(m['threshold_at_grid_edge'] for m in ms)
        row['best_epoch'] = [m['best_epoch'] for m in ms]
        row['train_min_per_epoch'] = mean_std([np.mean(h['epoch_seconds']) / 60 for h in hs])
        row['paper'] = {
            thr: {
                kind: {
                    k: mean_std([m['paper_patches'][thr][kind][k] for m in ms])
                    for k in ms[0]['paper_patches'][thr][kind]
                }
                for kind in ms[0]['paper_patches'][thr]
            }
            for thr in ('@thr', '@0.5')
        }
        summary[model] = row
    return summary


def _order(models) -> list[str]:
    return [m for m in (OURS,) if m in models] + sorted(m for m in models if m != OURS)


def _name(model: str) -> str:
    return f'**{model}** (ours)' if model == OURS else model


def _photo_table(runs: dict, order: list[str]) -> list[str]:
    """IoU of each test photograph at each run's threshold, averaged over seeds."""
    first = next(iter(runs[order[0]].values()))
    rust = first['counts'][:, [0, 2]].sum(1) / first['counts'].sum(1)
    lines = [
        '| Photograph | Camera | Rust | ' + ' | '.join(order) + ' |',
        '|---|---|---|' + '---|' * len(order),
    ]
    per_model = {m: np.mean([iou(runs[m][s]['counts']) for s in sorted(runs[m])], 0) for m in order}
    for i, (pid, cam) in enumerate(zip(first['photos'], first['cameras'], strict=True)):
        cells = ' | '.join(f'{per_model[m][i]:.4f}' for m in order)
        lines.append(f'| {pid} | {cam} | {100 * rust[i]:.2f}% | {cells} |')
    return lines


def _camera_table(runs: dict, order: list[str]) -> list[str]:
    """Pooled test IoU of each camera's photographs, averaged over seeds."""
    first = next(iter(runs[order[0]].values()))
    cams = sorted(set(first['cameras'].tolist()))
    lines = [
        '| Camera | Photographs | ' + ' | '.join(order) + ' |',
        '|---|---|' + '---|' * len(order),
    ]
    for cam in cams:
        sel = first['cameras'] == cam
        cells = ' | '.join(
            f'{np.mean([iou(runs[m][s]["counts"][sel].sum(0)) for s in runs[m]]):.4f}'
            for m in order
        )
        lines.append(f'| {cam} | {int(sel.sum())} | {cells} |')
    return lines


def speed_table(scale_dir: str, order: list[str], window: int, scale: float) -> list[str]:
    """Time per photograph from NWRD_final's T4 benchmark of the same architectures (D5)."""
    bench = _read_json(
        os.path.join(
            os.path.dirname(scale_dir),
            'nwrd_final',
            f'benchmark_{os.path.basename(scale_dir)}.json',
        )
    )
    if not bench:
        return []
    rows = {r['model']: r for r in bench['results']}
    per_photo = {
        k: len(grid(*scaled_shape(h, w, scale), window)) for k, (h, w) in PHOTO_SIZES.items()
    }
    lines = [
        '',
        f'## Speed ({bench["env"]["device"]}, measured by NWRD_final; results/nwrd_final/)',
        '',
        'Speed depends on the architecture and the window, not on the photographs, so it is not '
        'measured again. Seconds per photograph = windows per photograph / fp16 throughput at '
        'batch 8.',
        '',
        '| Model | Params (M) | GFLOPs per window | Windows/s | '
        + ' | '.join(f's per photograph: {k} ({n} windows)' for k, n in per_photo.items())
        + ' |',
        '|---|---|---|---|' + '---|' * len(per_photo),
    ]
    for m in (m for m in order if m in rows):
        r = rows[m]
        fps = r['gpu_throughput_img_s_bs8_fp16']
        lines.append(
            f'| {m} | {r["params_M"]:.1f} | {r["GFLOPs"]:.1f} | {fps:.0f} | '
            + ' | '.join(f'{n / fps:.1f}' for n in per_photo.values())
            + ' |'
        )
    return lines


def write_comparison(scale_dir: str, runs: dict, summary: dict, tests: dict) -> str:
    """Writes <scale_dir>/comparison.md and returns its path."""
    order = _order(summary)
    any_m = next(iter(next(iter(runs.values())).values()))['metrics']
    pp = any_m['paper_patches']
    L = [
        f"# CANet-B4 against the NWRD paper's baselines on the public NWRD release: "
        f'{os.path.basename(os.path.abspath(scale_dir))}',
        '',
        f'{any_m["window"]}px windows of photographs at {any_m["scale"]:g}× scale. Test: the '
        f"release's {any_m['test_photos']} test photographs, every pixel scored once at native "
        f'resolution ({100 * any_m["test_rust_pixel_fraction"]:.2f}% rust). Threshold chosen per '
        f'run on the {any_m["val_photos"]} validation photographs (IoU-maximising, grid '
        '0.01–0.99), then frozen for test. Metrics pooled over all test pixels; mean ± sample sd '
        'over seeds.',
        '',
        '## Accuracy (test photographs, validation-selected threshold)',
        '',
        '| Model | Seeds | ' + ' | '.join(METRICS) + ' | IoU @0.5 | Threshold |',
        '|---|---|' + '---|' * (len(METRICS) + 2),
    ]
    for m in order:
        r = summary[m]
        cells = [_fmt(*r['test@thr'][k], r['n']) for k in METRICS]
        thr = f'{r["threshold"][0]:.2f}' + (' ⚠ grid edge' if r['threshold_at_grid_edge'] else '')
        L.append(
            f'| {_name(m)} | {r["n"]} | '
            + ' | '.join(cells)
            + f' | {_fmt(*r["test@0.5"]["IoU"], r["n"])} | {thr} |'
        )
    L += [
        '',
        '## Each run',
        '',
        '| Model | Seed | Test IoU | Dice | Precision | Recall | IoU @0.5 | Val IoU | Threshold '
        '| Best epoch | Train min/epoch |',
        '|---|---|' + '---|' * 9,
    ]
    for m in order:
        for s in sorted(runs[m]):
            met, h = runs[m][s]['metrics'], runs[m][s]['history']['history']
            t = met['test@thr']
            L.append(
                f'| {m} | {s} | '
                + ' | '.join(f'{t[k]:.4f}' for k in ('IoU', 'Dice', 'Precision', 'Recall'))
                + f' | {met["test@0.5"]["IoU"]:.4f} | {met["val@thr"]["IoU"]:.4f} | '
                f'{met["threshold"]:.2f} | {met["best_epoch"]} | '
                f'{np.mean(h["epoch_seconds"]) / 60:.1f} |'
            )
    L += ['', "## Each test photograph (IoU at each run's threshold, mean over seeds)", '']
    L += _photo_table(runs, order)
    L += ['', '## By camera (pooled test IoU, mean over seeds)', '']
    L += _camera_table(runs, order)
    if tests:

        def ci(b):
            return f'[{b["ci95"][0]:+.4f}, {b["ci95"][1]:+.4f}] (p {b["p_two_sided"]:.4f})'

        L += [
            '',
            '## Is the difference real? (CANet-B4 minus baseline, test photographs)',
            '',
            '| Baseline | Paired seeds | ΔIoU | 95% CI, resampling photographs | ΔDice | '
            'ΔDice 95% CI | Seed-level t-test p | CANet-B4 higher (seeds) | Verdict |',
            '|---|---|---|---|---|---|---|---|---|',
        ]
        for base, t in tests.items():
            b, tt, n = t['bootstrap_photo'], t['seed_ttest'], len(t['seeds'])
            reading = f'CANet-B4 {VERDICTS[t["verdict"]]} {base}'
            if n < 3:
                reading += f' (interim: {n} of 3 seeds)'
            p_seeds = f'{tt["p"]:.4f}' if tt else 'n/a (1 seed)'
            L.append(
                f'| {base} | {n} | {b["IoU"]["delta"]:+.4f} | {ci(b["IoU"])} | '
                f'{b["Dice"]["delta"]:+.4f} | {ci(b["Dice"])} | {p_seeds} | '
                f'{t["canet_wins"]} of {n} | {reading} |'
            )
        L += [
            '',
            'Verdict: the decision rule fixed before any result existed (docs/protocol.md §5). '
            'CANet-B4 *improves on* a baseline if its mean test IoU is higher, the 95% CI of '
            'ΔIoU that resamples test photographs excludes 0, and it is higher on at least 2 of 3 '
            'seeds; it *is worse* if that CI lies below 0; otherwise the two are *comparable*. '
            'The interval captures test-set sampling, not training-run variance; the seed-level '
            't-test does, with little power at 3 seeds.',
        ]
    if pp['patches'] == PAPER_TEST_PATCHES:
        patches = (
            f'The {pp["patches"]:,} 128px patches of the half-size test photographs that the '
            f"paper's test.py scores ({pp['rust_patches']} with rust), in its order."
        )
    else:
        patches = (
            f"A partial test set: {pp['patches']:,} of the paper's {PAPER_TEST_PATCHES:,} test "
            f'patches ({pp["rust_patches"]} with rust), so the published rows are not comparable '
            'with ours.'
        )
    halved = 'resized by area to half size' if any_m['scale'] == 1 else 'taken at half size'
    L += [
        '',
        "## On the NWRD paper's own test patches",
        '',
        f'{patches} Our predictions are those scored above, {halved}; masks halved by area.',
        '',
        '| Result | Precision | Recall | F1 / Dice | IoU |',
        '|---|---|---|---|---|',
    ]
    for p in PAPER:
        s = p['scores']
        L.append(
            f'| Paper, {p["table"]}, UNet + APF, {p["patches"]} | '
            + ' | '.join(
                f'{s[k]:.3f}' if k in s else '–' for k in ('Precision', 'Recall', 'F1', 'IoU')
            )
            + ' |'
        )
    for m in order:
        for thr, kind, label in PAPER_ROWS:
            s = summary[m]['paper'][thr][kind]
            f1 = s['F1'] if 'F1' in s else s['Dice']  # the authors' averaging has no IoU
            iou_cell = f'{s["IoU"][0]:.3f}' if 'IoU' in s else '–'
            L.append(
                f'| {m}: {label} | {s["Precision"][0]:.3f} | {s["Recall"][0]:.3f} | '
                f'{f1[0]:.3f} | {iou_cell} |'
            )
    L += [
        '',
        "The paper's rows are as published, on the same photographs. Table 3 is its UNet trained "
        'with APF (focal loss, 500 epochs, 128px patches at stride 32) and scored by its test.py '
        'on all the patches. Table 4 is a separately trained UNet (stride 128, background-only '
        'patches removed from training as well as test, paper §5), and its IoU was not computed '
        "by test.py. The paper's training set held twins of test photographs 80 and 89 "
        '(docs/dataset.md); no model here trained on a twin of a validation or test photograph. '
        "The `unet` rows are the paper's architecture re-trained under this protocol, so "
        'CANet-B4 against `unet` is the like-for-like comparison.',
    ]
    L += speed_table(scale_dir, order, any_m['window'], any_m['scale'])
    L += ['', '## Models', ''] + [f'- `{m}` — {MODELS.get(m, m)}' for m in order]
    path = os.path.join(scale_dir, 'comparison.md')
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(L) + '\n')
    return path


def write_model_summary(out_dir: str, model: str) -> str | None:
    """Writes <out_dir>/<model>/summary.md: one model's runs, seed by seed, and its mean ± sd.
    Returns its path, or None while the model has no finished run."""
    runs = load_runs(out_dir).get(model)
    if not runs:
        return None
    r = summarise({model: runs})[model]
    keys = ('IoU', 'Dice', 'Precision', 'Recall')
    any_m = next(iter(runs.values()))['metrics']
    L = [
        f'# {model}: {MODELS.get(model, model)}',
        '',
        f'{any_m["window"]}px windows at {any_m["scale"]:g}× scale. Test: '
        f'{any_m["test_photos"]} photographs, every pixel scored once at native resolution. '
        'Threshold chosen per run on validation; metrics pooled over all test pixels. The paper '
        "patches column is F1 on the NWRD paper's test patches with the authors' averaging at "
        '0.5.',
        '',
        '| Seed | Test IoU | Dice | Precision | Recall | IoU @0.5 | Val IoU | Threshold | '
        'Best epoch | Paper patches F1 | Train min/epoch |',
        '|---|' + '---|' * 10,
    ]
    for s in sorted(runs):
        m, h = runs[s]['metrics'], runs[s]['history']['history']
        t = m['test@thr']
        L.append(
            f'| {s} | '
            + ' | '.join(f'{t[k]:.4f}' for k in keys)
            + f' | {m["test@0.5"]["IoU"]:.4f} | {m["val@thr"]["IoU"]:.4f} | '
            f'{m["threshold"]:.2f} | {m["best_epoch"]} | '
            f'{m["paper_patches"]["@0.5"]["authors"]["F1"]:.4f} | '
            f'{np.mean(h["epoch_seconds"]) / 60:.1f} |'
        )
    means = [r['test@thr'][k] for k in keys] + [r['test@0.5']['IoU'], r['val@thr']['IoU']]
    L.append(
        '| **mean ± sd** | '
        + ' | '.join(_fmt(*v, r['n']) for v in means)
        + f' | {r["threshold"][0]:.2f} | – | '
        f'{_fmt(*r["paper"]["@0.5"]["authors"]["F1"], r["n"])} | '
        f'{r["train_min_per_epoch"][0]:.1f} |'
    )
    L += [
        '',
        "Side by side with the other models: comparison.md of the scale's results folder "
        '(python -m nwrd_public.compare results).',
    ]
    path = os.path.join(out_dir, model, 'summary.md')
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(L) + '\n')
    return path


def plot_training_curves(runs: dict, path: str) -> None:
    """Validation loss and IoU per epoch: mean over seeds, band from min to max."""
    import matplotlib

    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for model in _order(runs):
        seeds = runs[model]
        for key, ax in (('val_loss', axes[0]), ('val_iou', axes[1])):
            curves = [seeds[s]['history']['history'][key] for s in sorted(seeds)]
            n = min(map(len, curves))
            arr = np.array([c[:n] for c in curves])
            x = np.arange(1, n + 1)
            (line,) = ax.plot(x, arr.mean(0), label=model, lw=2 if model == OURS else 1.4)
            if len(curves) > 1:
                ax.fill_between(x, arr.min(0), arr.max(0), color=line.get_color(), alpha=0.15)
    axes[0].set_title('Validation loss (windows)')
    axes[1].set_title('Validation IoU (whole photographs, threshold 0.5)')
    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def compare(scale_dir: str, log=print) -> dict | None:
    """Writes comparison.md, comparison.json, the training curves and each model's summary."""
    runs = load_runs(scale_dir)
    if not runs:
        log(f'[compare] no finished runs in {scale_dir}')
        return None
    incomplete = [
        r['dir']
        for seeds in runs.values()
        for r in seeds.values()
        if not {'history', 'counts', 'paper'} <= set(r)
    ]
    if incomplete:
        raise FileNotFoundError(
            f'history.json, test_counts.npz or paper_patches.npz missing in {incomplete}'
        )
    summary = summarise(runs)
    tests = significance(runs)
    plot_training_curves(runs, os.path.join(scale_dir, 'figures', 'training_curves.png'))
    md_path = write_comparison(scale_dir, runs, summary, tests)
    for model in runs:
        write_model_summary(scale_dir, model)
    with open(os.path.join(scale_dir, 'comparison.json'), 'w') as f:
        json.dump({'summary': summary, 'significance': tests}, f, indent=2)
    with open(md_path, encoding='utf-8') as f:
        log(f.read())
    return summary


SUMMARY_COLUMNS = (
    'scale', 'model', 'seed', 'best_epoch', 'threshold', 'threshold_at_grid_edge',
    'test_iou', 'test_dice', 'test_precision', 'test_recall', 'test_specificity',
    'test_accuracy', 'test_iou_at_0.5', 'val_iou',
    'paper_authors_precision_at_0.5', 'paper_authors_recall_at_0.5', 'paper_authors_f1_at_0.5',
    'paper_authors_rust_only_f1_at_0.5', 'paper_pooled_iou', 'paper_rust_only_iou',
    'train_min_per_epoch',
)  # fmt: skip


def summary_rows(results_root: str) -> list[dict]:
    """One row per (scale, model, seed) found under `results_root`."""
    rows = []
    for scale in SCALES:
        runs = load_runs(os.path.join(results_root, scale))
        for model in _order(runs):
            for seed in sorted(runs[model]):
                m, h = runs[model][seed]['metrics'], runs[model][seed].get('history')
                t, p = m['test@thr'], m['paper_patches']
                rows.append(
                    {
                        'scale': scale,
                        'model': model,
                        'seed': seed,
                        'best_epoch': m['best_epoch'],
                        'threshold': m['threshold'],
                        'threshold_at_grid_edge': m['threshold_at_grid_edge'],
                        **{f'test_{k.lower()}': t[k] for k in METRICS},
                        'test_iou_at_0.5': m['test@0.5']['IoU'],
                        'val_iou': m['val@thr']['IoU'],
                        'paper_authors_precision_at_0.5': p['@0.5']['authors']['Precision'],
                        'paper_authors_recall_at_0.5': p['@0.5']['authors']['Recall'],
                        'paper_authors_f1_at_0.5': p['@0.5']['authors']['F1'],
                        'paper_authors_rust_only_f1_at_0.5': p['@0.5']['authors_rust_only']['F1'],
                        'paper_pooled_iou': p['@thr']['pooled']['IoU'],
                        'paper_rust_only_iou': p['@thr']['rust_only']['IoU'],
                        'train_min_per_epoch': (
                            float(np.mean(h['history']['epoch_seconds']) / 60) if h else ''
                        ),
                    }
                )
    return rows


def write_summary_csv(results_root: str) -> str:
    """results/summary.csv: one row per (scale, model, seed), for a spreadsheet."""
    path = os.path.join(results_root, 'summary.csv')
    rows = [
        {k: round(v, 6) if isinstance(v, float) else v for k, v in row.items()}
        for row in summary_rows(results_root)
    ]
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=SUMMARY_COLUMNS)
        w.writeheader()
        w.writerows(rows)
    return path


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter
    )
    ap.add_argument('results', help='the results folder holding 512px/ and 128px/')
    a = ap.parse_args(argv)
    for scale in SCALES:
        if os.path.isdir(os.path.join(a.results, scale)):
            compare(os.path.join(a.results, scale))
    print('wrote', write_summary_csv(a.results))


if __name__ == '__main__':
    # The reports contain ± and ≈, which a redirected stdout on Windows (cp1252) cannot encode.
    sys.stdout.reconfigure(encoding='utf-8')
    main()


In [ ]:
%%writefile code/nwrd_public/run.py
"""Runs one model at one scale: cut -> train -> eval -> summary.

    python -m nwrd_public.run --model canet_b4 --preset 512px --data-root data/raw/NWRD --out runs
    python -m nwrd_public.run --model unet --preset 128px --out runs   # on Kaggle

On Kaggle the release is found under /kaggle/input. It is checked against data/manifest.json
before anything else, and the run stops if any file differs. Finished stages are skipped, and an
interrupted training run resumes from last.pt, so the same command can be repeated across Kaggle
sessions until everything is done. Exit code 3 means the time budget stopped training: run the
same command again to continue.

The side-by-side comparison of the three models: python -m nwrd_public.compare results
"""

import argparse
import json
import os
import platform
import shutil
import sys
import tempfile
import time
import traceback
import typing
from dataclasses import fields

from . import __version__
from .config import PRESETS, Config

STAGES = ('cut', 'train', 'eval', 'summary')
# Stages that create run records. Only these write config.json and log.txt, so re-writing a
# finished run's summary never overwrites what the run recorded.
RECORDING_STAGES = {'cut', 'train', 'eval'}
# Settings that may change between the sessions of one experiment: where things are, how many
# workers, the time budget and which seeds this session runs. Every other setting must match the
# experiment's config.json, since a seed's folder does not record its window or scale.
SESSION_FIELDS = {
    'data_root', 'manifest', 'out_dir', 'window_dir', 'num_workers', 'time_budget_hours', 'seeds',
}  # fmt: skip


def _parse_bool(s: str) -> bool:
    v = s.strip().lower()
    if v in ('1', 'true', 'yes'):
        return True
    if v in ('0', 'false', 'no'):
        return False
    raise argparse.ArgumentTypeError(f'expected true or false, got {s!r}')


def parse_args(argv=None) -> argparse.Namespace:
    """Every Config field is a flag; the preset's values are the defaults."""
    pre = argparse.ArgumentParser(add_help=False)
    pre.add_argument('--preset', choices=sorted(PRESETS), default='512px')
    preset = pre.parse_known_args(argv)[0].preset
    defaults = {**Config().to_dict(), **PRESETS[preset]}
    hints = typing.get_type_hints(Config)

    p = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter
    )
    p.add_argument(
        '--preset',
        choices=sorted(PRESETS),
        default='512px',
        help='the scale whose window size and resize factor are the defaults (config.py)',
    )
    p.add_argument('--stage', default='all', choices=('all',) + STAGES)
    p.add_argument('--out', dest='out_dir', default=defaults['out_dir'])
    for f in fields(Config):
        if f.name == 'out_dir':
            continue
        flag, hint, default = '--' + f.name.replace('_', '-'), hints[f.name], defaults[f.name]
        if hint is bool:
            p.add_argument(
                flag, dest=f.name, type=_parse_bool, default=default, metavar='{true,false}'
            )
        elif typing.get_origin(hint) is list:
            p.add_argument(
                flag, dest=f.name, nargs='+', type=typing.get_args(hint)[0], default=default
            )
        else:
            p.add_argument(flag, dest=f.name, type=hint, default=default)
    return p.parse_args(argv)


def _environment() -> str:
    """Library versions and device, recorded at the top of every session's log."""
    from importlib.metadata import PackageNotFoundError, version

    import cv2
    import numpy
    import torch

    def v(package):
        try:
            return version(package)
        except PackageNotFoundError:
            return 'not installed'

    device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    return (
        f'python {platform.python_version()}, torch {torch.__version__}, '
        f'numpy {numpy.__version__}, opencv {cv2.__version__}, '
        f'albumentations {v("albumentations")}, '
        f'segmentation-models-pytorch {v("segmentation_models_pytorch")}, '
        f'efficientnet-pytorch {v("efficientnet_pytorch")}, device {device}'
    )


def default_window_dir(cfg: Config) -> str:
    """Training windows are cut outside out_dir, so they never enter a run's saved output."""
    return os.path.join(
        tempfile.gettempdir(), 'nwrd_public_windows', f'{cfg.window}px_x{cfg.scale:g}'
    )


def check_same_experiment(cfg: Config) -> None:
    """Refuses to continue the runs in cfg.out_dir if they were made with other settings: they
    would be skipped as finished, reported and collected as this experiment's."""
    recorded = os.path.join(cfg.out_dir, 'config.json')
    if not os.path.exists(recorded):
        return
    with open(recorded) as f:
        done = json.load(f)
    differ = {
        k: (done.get(k), v)
        for k, v in cfg.to_dict().items()
        if k not in SESSION_FIELDS and done.get(k) != v
    }
    if differ:
        raise RuntimeError(
            f'{cfg.out_dir} holds runs made with other settings (recorded, now): {differ}; '
            'use another --out'
        )


def main(argv=None) -> int:
    t_session = time.time()  # the time budget counts from here, across all runs
    a = parse_args(argv)
    cfg = Config(**{f.name: getattr(a, f.name) for f in fields(Config)})
    cfg.validate()
    stages = STAGES if a.stage == 'all' else (a.stage,)
    recording = bool(RECORDING_STAGES & set(stages))
    if recording:
        check_same_experiment(cfg)
    os.makedirs(cfg.out_dir, exist_ok=True)
    log_file = (
        open(os.path.join(cfg.out_dir, 'log.txt'), 'a', encoding='utf-8') if recording else None
    )

    def log(msg: str) -> None:
        line = f'{time.strftime("%H:%M:%S")} {msg}'
        print(line, flush=True)
        if log_file:
            log_file.write(line + '\n')
            log_file.flush()

    try:
        return _run(cfg, a.preset, stages, log, t_session, recording)
    except Exception:
        # On Kaggle the notebook's own output cannot be downloaded, but runs/log.txt can.
        if log_file:
            log_file.write(f'{time.strftime("%H:%M:%S")} CRASHED:\n{traceback.format_exc()}')
        raise
    finally:
        if log_file:
            log_file.close()


def _run(cfg: Config, preset: str, stages: tuple, log, t_session: float, recording: bool) -> int:
    log(
        f'nwrd_public {__version__} | {cfg.model} | preset {preset} | '
        f'stages {",".join(stages)} | {_environment()}'
    )
    release = windows = None
    if {'cut', 'train', 'eval'} & set(stages):
        from .dataset import open_release

        release = open_release(cfg.data_root or None, cfg.manifest or None, cfg.limit, log)
        cfg.data_root = release.root
    if {'cut', 'train'} & set(stages):
        from .windows import cut_training_windows

        windows = cut_training_windows(
            release,
            release.splits['train'],
            cfg.window_dir or default_window_dir(cfg),
            cfg.window,
            cfg.scale,
            cfg.bg_keep_ratio,
            cfg.data_seed,
            workers=max(1, cfg.num_workers),
            log=log,
        )
        shutil.copy(
            os.path.join(windows['dir'], 'windows.json'), os.path.join(cfg.out_dir, 'windows.json')
        )
    if recording:
        with open(os.path.join(cfg.out_dir, 'config.json'), 'w') as f:
            json.dump({'preset': preset, **cfg.to_dict()}, f, indent=2)

    complete, session_start = True, None
    for seed in cfg.seeds:
        run_dir = os.path.join(cfg.out_dir, cfg.model, f'seed{seed}')
        if 'train' in stages and not os.path.exists(os.path.join(run_dir, 'TRAINED')):
            from .train import train_run

            if not train_run(
                cfg, release, windows, seed, run_dir, log=log, session_start=session_start
            ):
                complete = False
                break  # out of time budget: stop here, resume next session
            session_start = t_session
        if (
            'eval' in stages
            and os.path.exists(os.path.join(run_dir, 'TRAINED'))
            and not os.path.exists(os.path.join(run_dir, 'metrics.json'))
        ):
            from .evaluate import evaluate_run

            evaluate_run(cfg, release, run_dir, log=log)

    if 'summary' in stages:
        from .compare import write_model_summary

        path = write_model_summary(cfg.out_dir, cfg.model)
        if path:
            with open(path, encoding='utf-8') as f:
                log(f.read())

    if not complete:
        log(
            'INCOMPLETE: continue in a new session with these runs/ in place (on Kaggle: '
            'scripts/run_on_kaggle.py --resume, which attaches them).'
        )
    return 0 if complete else 3


if __name__ == '__main__':
    # The reports contain ±, which a redirected stdout on Windows (cp1252) cannot encode.
    sys.stdout.reconfigure(encoding='utf-8')
    sys.exit(main())


### Resume

In [ ]:
# Continue an earlier session. Kaggle offers the output of a finished version as an input
# (not of one killed at the session limit); its runs/ is copied into place here, once it is
# checked to be this notebook's model and scale.
prev = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/runs/config.json', recursive=True))
for d in prev:
    found = json.load(open(os.path.join(d, 'config.json')))
    if (found.get('model'), found.get('preset')) != (MODEL, PRESET):
        raise RuntimeError("the attached %s holds runs of %s %s, not %s %s: attach this kernel's own output"
                           % (d, found.get('model'), found.get('preset'), MODEL, PRESET))
if prev and not os.path.exists(os.path.join(OUT, 'config.json')):
    shutil.copytree(prev[0], OUT, dirs_exist_ok=True)
    print('restored previous runs from', prev[0])
    with open(os.path.join(OUT, 'log.txt'), 'a', encoding='utf-8') as f:
        f.write('--- new session: restored previous runs from %s\n' % prev[0])
states = {}
for d in sorted(glob.glob(os.path.join(OUT, '*', 'seed*'))):
    states[d] = ('done' if os.path.exists(os.path.join(d, 'metrics.json')) else
                 'trained' if os.path.exists(os.path.join(d, 'TRAINED')) else
                 'partial' if os.path.exists(os.path.join(d, 'last.pt')) else 'empty')
    print(' ', os.path.relpath(d, OUT), states[d])
print('%d run directories found' % len(states))
if EXPECT_RESTORE and not any(s != 'empty' for s in states.values()):
    print('/kaggle/input:', sorted(glob.glob('/kaggle/input/*/*/*'))[:30])
    raise RuntimeError('Resume was requested but no earlier runs/ was found under /kaggle/input. '
                       'Stopping now instead of retraining every run from scratch.')

### Run

In [ ]:
# The release is found under /kaggle/input and checked against code/data/manifest.json first.
cmd = [sys.executable, '-m', 'nwrd_public.run', '--model', MODEL, '--preset', PRESET, '--out', OUT,
       '--time-budget-hours', str(KAGGLE_TIME_BUDGET_HOURS), '--num-workers', str(NUM_WORKERS)]
print(' '.join(cmd))
proc = subprocess.Popen(cmd, cwd='code', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
code = proc.wait()
print('exit code', code, '(3 = stopped at the time budget: continue with scripts/run_on_kaggle.py --model',
      MODEL, '--preset', PRESET, "--resume, or attach this version's output as an input and commit again)")
# Fail the version, so Kaggle reports ERROR instead of a COMPLETE run that trained nothing.
if code not in (0, 3):
    raise RuntimeError('the pipeline failed (exit code %d); the traceback is in runs/log.txt' % code)

In [ ]:
from IPython.display import Markdown, display
summary = os.path.join(OUT, MODEL, 'summary.md')
if os.path.exists(summary):
    display(Markdown(open(summary, encoding='utf-8').read()))

## Taking the results back to the repository

`python scripts/run_on_kaggle.py --model <model> [--preset 128px] --status` downloads this version's output to `runs_kaggle/<scale>/<model>/<account>_v<N>/` (weights stay on Kaggle). Then `python scripts/collect_results.py runs_kaggle/<scale>/<model>/<account>_v<N>` copies its runs into `results/<scale>/<model>/` and rebuilds the side-by-side comparison of the models collected so far: `results/<scale>/comparison.md` and `results/summary.csv`.